    # Stage 2 Optuna: 11-feature minimal-state median-return objective + constraints

Цель Stage 2 — не менять execution policy и не менять feature sets, а сделать более аккуратный повтор HPO с:
- 11-мерным контекстом `10 market features + state_in_position` и суженным диапазоном памяти `[250, 400]` баров;
- objective, согласованным с исходной robust return logic: `median over seeds of median over symbols return_pct`;
- дополнительными hard constraints по активности, числу сделок, profit factor и max drawdown;
- Calmar-like ratio, turnover, memory horizon и mean-return metrics как diagnostics/tie-breakers, а не как основной objective.

Execution layer остаётся фиксированным, чтобы сравнение `corr` vs `hybrid` не смешивалось с дополнительной оптимизацией `min_hold`, `cooldown` или `confidence_threshold`.

Stage 2 hard constraints:
- `all_symbols_active_share >= 1.0`;
- `all_symbols_have_long_share >= 1.0`;
- `MIN_TOTAL_TRADES < median_total_trades < MAX_TOTAL_TRADES`;
- `median_profit_factor > MIN_PROFIT_FACTOR`;
- `median_max_drawdown_abs < MAX_DRAWDOWN_ABS`.

Calmar-like ratio сохраняется как diagnostic для анализа risk-adjusted качества, но Optuna максимизирует именно `median_of_seed_median_return_pct` среди valid trials.


In [1]:
import json
import sys
import math
import gc
from pathlib import Path
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd

try:
    import optuna
except ImportError as exc:
    raise ImportError("Optuna не установлен. Установи: pip install optuna") from exc

PROJECT_ROOT = Path(r"D:\PythonProjects\VKR")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_processing.functions.klines_dataloader import KlinesDataLoader
from data_processing.functions.stream_indicators import stream_TA_lib
from data_processing.functions.transform_indicators import transform_indicators_df
from data_processing.functions.rolling_z_score_clip import rolling_z_score_clip_df
from backtest.backtest_class import Backtesting

print("Optuna version:", optuna.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)

Optuna version: 4.8.0
PROJECT_ROOT: D:\PythonProjects\VKR


D:\PythonProjects\VKR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Основные настройки эксперимента

Меняй только блок `RUN_PRESET` / `ALGORITHMS_TO_RUN`, если запускаешь notebook на разных компьютерах.

In [2]:
# =====================================================================
# Machine split
# =====================================================================
MACHINE_PRESETS = {
    # Recommended split: one TS + one UCB on each computer.
    "A_discounted": ["discounted_lints", "discounted_linucb"],
    "B_sliding": ["sw_lints", "sw_linucb"],
    "ALL": ["discounted_lints", "discounted_linucb", "sw_lints", "sw_linucb"],
}

RUN_PRESET = "A_discounted"  # <-- на втором компьютере поставь "B_sliding"
ALGORITHMS_TO_RUN = MACHINE_PRESETS[RUN_PRESET]

# Optional suffix to avoid overwriting outputs from different machines/runs.
RUN_LABEL = f"{RUN_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# =====================================================================
# Core protocol
# =====================================================================
BASE_SEED = 142
OPTUNA_SAMPLER_SEED = 20260515
ALL_SYMBOLS = ["BTCUSDT", "DOGEUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT"]
ACTIONS = [0, 1]

INTERVAL = 240
HORIZON = 10
TRADE_COST = 0.0025
OHLCV_RELATIVE_PATH = Path("data/klines_data/crypto_240m_bybit_TEST.parquet")

VAL_BARS = 2000
TEST_BARS = 2000
EMBARGO_BARS = 0

STATE_FEATURES = [
    "state_in_position",
]
META_COLS = ["timestamp", "symbol", "open", "high", "low", "close", "volume"]

# =====================================================================
# Feature-selection artifacts from screening stage
# =====================================================================
FEATURE_SELECTION_ROOT = PROJECT_ROOT / "feature_selection" / "hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2"
# Minimal-state screening artifacts. Keep only state1 directories here; do not fallback to old 13-feature screening.
SCREENING_DIR_CANDIDATES = [
    FEATURE_SELECTION_ROOT / "algorithm_specific_screening_defaults_state1_hmem325_ts30",
    FEATURE_SELECTION_ROOT / "algorithm_specific_screening_defaults_state1_hmem250_ts30",
]
SCREENING_DIR = next(
    (p for p in SCREENING_DIR_CANDIDATES if (p / "selected_feature_sets_for_optuna_balanced.json").exists()),
    SCREENING_DIR_CANDIDATES[0],
)
SELECTED_FOR_OPTUNA_JSON = SCREENING_DIR / "selected_feature_sets_for_optuna_balanced.json"

# No mixed set will be loaded. We use only selected corr/hybrid candidates.

# =====================================================================
# Local outputs; no database is used.
# =====================================================================
# Stage 2 outputs are written next to this notebook / inside hyperparameters_optimization,
# not into feature_selection outputs. Feature-selection artifacts are still read from FEATURE_SELECTION_ROOT.
HYPERPARAMETERS_ROOT = PROJECT_ROOT / "hyperparameters_optimization"
OUTPUT_ROOT = HYPERPARAMETERS_ROOT / "stage2_optuna_state1_median_return_constraints_func_memory250_400_local"
OUTPUT_DIR = OUTPUT_ROOT / RUN_LABEL
DIAGNOSTICS_DIR = OUTPUT_DIR / "diagnostics"
STUDIES_DIR = OUTPUT_DIR / "studies"
FEATURE_CACHE_DIR = HYPERPARAMETERS_ROOT / "stage2_optuna_feature_cache"

for d in [OUTPUT_DIR, DIAGNOSTICS_DIR, STUDIES_DIR, FEATURE_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =====================================================================
# Fixed execution settings during Optuna.
# These are not tuned in Stage 2.
# =====================================================================
START_CAPITAL = 100.0
POSITION_SIZE = 0.10
MIN_HOLD_BARS = 4
COOLDOWN_BARS = 2
CONFIDENCE_THRESHOLD = 0.005
UPDATE_ON_VALIDATION = True

# =====================================================================
# Validation split diagnostics.
# Backtest still runs sequentially over the full validation period with
# update_on_validation=True. These settings only add post-hoc diagnostics
# for the first and second validation halves; they do not change the
# primary Optuna objective or feasibility constraints.
# =====================================================================
SPLIT_VAL_HALVES_FOR_DIAGNOSTICS = True
VAL_HALF_MODE_NAMES = ("val_first", "val_second")

# =====================================================================
# Fixed reward clipping; not optimized.
# =====================================================================
REWARD_CLIP = 0.10

# =====================================================================
# Optuna budget.
# Stage 2 minimal-state: memory range is narrowed to [250, 400], so 40 trials per study are sufficient.
# TS is stochastic, so each trial is evaluated across 5 fixed seeds in full mode.
# UCB is deterministic enough; 1 seed is used.
# =====================================================================
FULL_N_TRIALS = 40
FULL_TS_SEEDS_PER_TRIAL = [3142, 3143, 3144, 3145, 3146]
FULL_UCB_SEEDS_PER_TRIAL = [3142]

# Smoke test mode can be enabled for a short feasibility run.
# Run it first to verify that trade-count constraints allow at least some valid trials.
# After the smoke test is successful, set SMOKE_TEST = False and restart the notebook for the full run.
SMOKE_TEST = False
SMOKE_N_TRIALS = 16
SMOKE_TS_SEEDS_PER_TRIAL = [3142, 3143]
SMOKE_UCB_SEEDS_PER_TRIAL = [3142]

if SMOKE_TEST:
    N_TRIALS = SMOKE_N_TRIALS
    TS_SEEDS_PER_TRIAL = SMOKE_TS_SEEDS_PER_TRIAL
    UCB_SEEDS_PER_TRIAL = SMOKE_UCB_SEEDS_PER_TRIAL
else:
    N_TRIALS = FULL_N_TRIALS
    TS_SEEDS_PER_TRIAL = FULL_TS_SEEDS_PER_TRIAL
    UCB_SEEDS_PER_TRIAL = FULL_UCB_SEEDS_PER_TRIAL

# =====================================================================
# Constrained median-return objective settings.
# Objective for valid trials:
#   median_of_seed_median_return_pct = median_seed(median_symbol(return_pct)).
#
# Calmar-like ratio is retained as a diagnostic, not as the Optuna objective.
# Hard constraints:
#   all_symbols_active_share >= 1.0
#   all_symbols_have_long_share >= 1.0
#   MIN_TOTAL_TRADES < median_total_trades < MAX_TOTAL_TRADES
#   median_profit_factor > MIN_PROFIT_FACTOR
#   median_max_drawdown_abs < MAX_DRAWDOWN_ABS
#
# Infeasible trials keep their real objective; feasibility is handled by Optuna constraints_func.
# INVALID_SCORE is used only as a fallback for errors/non-finite objective values.
# =====================================================================
INVALID_SCORE = -1_000_000.0
MIN_TOTAL_TRADES = 100
MAX_TOTAL_TRADES = 650
MIN_PROFIT_FACTOR = 0.85
MAX_DRAWDOWN_ABS = 15.0
# Prevent artificially huge Calmar values when drawdown is extremely small.
CALMAR_DRAWDOWN_FLOOR_ABS = 1.0

# Enqueue the minimal-state screening default as trial 0 in every study.
# This is valid because screening was re-created under the same 11-feature state1 setup.
# IMPORTANT: keep these defaults aligned with the screening notebook used to select feature sets.
SCREENING_DEFAULT_MEMORY_HORIZON_BARS = 325
SCREENING_DEFAULT_LAMBDA_PRIOR = 1.0
SCREENING_DEFAULT_NOISE_STD = 0.03
SCREENING_DEFAULT_UCB_ALPHA = 0.10

ENQUEUE_DEFAULT_TRIAL = True

# =====================================================================
# Thompson Sampling confirmation stage.
# After main Optuna, top-3 TS trials per study are re-evaluated on 25 fresh seeds.
# This does not search new parameters; it only checks seed robustness of top trials.
# In smoke mode confirmation is disabled because the smoke test only checks feasibility.
# =====================================================================
RUN_TS_CONFIRMATION = False if SMOKE_TEST else True
TOP_N_CONFIRMATION = 3
CONFIRMATION_SEEDS = list(range(8000, 8025))

# =====================================================================
# Indicator config must match feature-selection/screening pipeline.
# =====================================================================
CONFIG_FOR_INDICATORS = {
    "ema_periods": [9, 21, 50, 100, 200],
    "momentum_indicators_periods": [14, 30, 72],
    "return_indicators_periods": [6, 24, 72, 168],
    "volatility_indicators_periods": [24, 72, 168],
    "level_periods": [24, 72, 168],
    "vol_ma_period": [24, 72, 168],
    "range_ma_period": [24, 72, 168],
}

print("SMOKE_TEST:", SMOKE_TEST)
print("RUN_LABEL:", RUN_LABEL)
print("ALGORITHMS_TO_RUN:", ALGORITHMS_TO_RUN)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SELECTED_FOR_OPTUNA_JSON:", SELECTED_FOR_OPTUNA_JSON)
print("STATE_FEATURES:", STATE_FEATURES)
print("N_STATE_FEATURES:", len(STATE_FEATURES))
print("SPLIT_VAL_HALVES_FOR_DIAGNOSTICS:", SPLIT_VAL_HALVES_FOR_DIAGNOSTICS)
print("VAL_HALF_MODE_NAMES:", VAL_HALF_MODE_NAMES)
print("MIN_TOTAL_TRADES:", MIN_TOTAL_TRADES)
print("MAX_TOTAL_TRADES:", MAX_TOTAL_TRADES)
print("MIN_PROFIT_FACTOR:", MIN_PROFIT_FACTOR)
print("MAX_DRAWDOWN_ABS:", MAX_DRAWDOWN_ABS)
print("CALMAR_DRAWDOWN_FLOOR_ABS:", CALMAR_DRAWDOWN_FLOOR_ABS)
print("RUN_TS_CONFIRMATION:", RUN_TS_CONFIRMATION)
print("ENQUEUE_DEFAULT_TRIAL:", ENQUEUE_DEFAULT_TRIAL)
print("N_TRIALS:", N_TRIALS)
print("TS_SEEDS_PER_TRIAL:", TS_SEEDS_PER_TRIAL)
print("UCB_SEEDS_PER_TRIAL:", UCB_SEEDS_PER_TRIAL)
print("TOP_N_CONFIRMATION:", TOP_N_CONFIRMATION)
print("N_CONFIRMATION_SEEDS:", len(CONFIRMATION_SEEDS))


SMOKE_TEST: False
RUN_LABEL: A_discounted_20260518_193538
ALGORITHMS_TO_RUN: ['discounted_lints', 'discounted_linucb']
OUTPUT_DIR: D:\PythonProjects\VKR\hyperparameters_optimization\stage2_optuna_state1_median_return_constraints_func_memory250_400_local\A_discounted_20260518_193538
SELECTED_FOR_OPTUNA_JSON: D:\PythonProjects\VKR\feature_selection\hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2\algorithm_specific_screening_defaults_state1_hmem250_ts30\selected_feature_sets_for_optuna_balanced.json
STATE_FEATURES: ['state_in_position']
N_STATE_FEATURES: 1
SPLIT_VAL_HALVES_FOR_DIAGNOSTICS: True
VAL_HALF_MODE_NAMES: ('val_first', 'val_second')
MIN_TOTAL_TRADES: 100
MAX_TOTAL_TRADES: 650
MIN_PROFIT_FACTOR: 0.85
MAX_DRAWDOWN_ABS: 15.0
CALMAR_DRAWDOWN_FLOOR_ABS: 1.0
RUN_TS_CONFIRMATION: True
ENQUEUE_DEFAULT_TRIAL: True
N_TRIALS: 40
TS_SEEDS_PER_TRIAL: [3142, 3143, 3144, 3145, 3146]
UCB_SEEDS_PER_TRIAL: [3142]
TOP_N_CONFIRMATION: 3
N_CONFIRMATION_SEEDS: 25


## 2. Диапазоны Optuna

Search space в Stage 2 меняет только bandit hyperparameters, execution параметры остаются фиксированными.

Используются диапазоны:

- discounted memory: `one_minus_gamma ∈ [0.0025, 0.0040]`, что соответствует `H_mem = 1/(1-gamma) ∈ [250, 400]` баров;
- sliding window: `window_size ∈ [250, 400]` с шагом 25;
- regularization: `lambda_prior ∈ [0.3, 10.0]` в log-scale;
- Thompson Sampling exploration: `noise_std ∈ [0.003, 0.04]` в log-scale;
- UCB exploration: `ucb_alpha ∈ [0.02, 0.30]` в log-scale.

`window_size` намеренно ограничен диапазоном 200–400 баров, чтобы Stage 2 проверял более короткую память и более быструю адаптацию к regime shift.


Контекст в этой версии: 10 market features + 1 state feature (`state_in_position`) = 11 total features.


In [3]:
# Search spaces.
# You can edit here, but keep ranges centered around screening defaults.
SEARCH_SPACE = {
    "discounted": {
        # gamma = 1 - one_minus_gamma.
        # H_mem = 1/(1-gamma) = 1/one_minus_gamma in [250, 400] bars.
        # one_minus_gamma_low=1/400=0.0025, high=1/250=0.0040.
        "one_minus_gamma_low": 0.0025,
        "one_minus_gamma_high": 0.0040,
        "lambda_prior_low": 0.3,
        "lambda_prior_high": 10.0,
    },
    "sliding": {
        # Minimal-state Stage 2 memory range: 250-400 bars.
        "window_size_low": 250,
        "window_size_high": 400,
        "window_size_step": 25,
        "lambda_prior_low": 0.3,
        "lambda_prior_high": 10.0,
    },
    "ts": {
        "noise_std_low": 0.003,
        "noise_std_high": 0.04,
    },
    "ucb": {
        "ucb_alpha_low": 0.02,
        "ucb_alpha_high": 0.30,
    },
}

TS_BANDITS = {"discounted_lints", "sw_lints"}
UCB_BANDITS = {"discounted_linucb", "sw_linucb"}
DISCOUNTED_BANDITS = {"discounted_lints", "discounted_linucb"}
SLIDING_BANDITS = {"sw_lints", "sw_linucb"}

BANDIT_TYPE_MAP = {
    "discounted_lints": "discounted_lints",
    "discounted_linucb": "discounted_linucb",
    "sw_lints": "sw_lints",
    "sw_linucb": "sw_linucb",
}

EXPECTED_MAIN_BACKTESTS_THIS_MACHINE = 0
EXPECTED_CONFIRMATION_BACKTESTS_THIS_MACHINE = 0
for algo in ALGORITHMS_TO_RUN:
    seeds = TS_SEEDS_PER_TRIAL if algo in TS_BANDITS else UCB_SEEDS_PER_TRIAL
    EXPECTED_MAIN_BACKTESTS_THIS_MACHINE += 2 * N_TRIALS * len(seeds)  # 2 feature sets per algorithm
    if RUN_TS_CONFIRMATION and algo in TS_BANDITS:
        EXPECTED_CONFIRMATION_BACKTESTS_THIS_MACHINE += 2 * TOP_N_CONFIRMATION * len(CONFIRMATION_SEEDS)

EXPECTED_MAIN_BACKTESTS_FULL = (
    2 * 2 * N_TRIALS * len(TS_SEEDS_PER_TRIAL)
    + 2 * 2 * N_TRIALS * len(UCB_SEEDS_PER_TRIAL)
)
EXPECTED_CONFIRMATION_BACKTESTS_FULL = (
    2 * 2 * TOP_N_CONFIRMATION * len(CONFIRMATION_SEEDS)
    if RUN_TS_CONFIRMATION else 0
)

print("N_TRIALS per study:", N_TRIALS)
print("Expected main Optuna backtests on this machine:", EXPECTED_MAIN_BACKTESTS_THIS_MACHINE)
print("Expected TS confirmation backtests on this machine:", EXPECTED_CONFIRMATION_BACKTESTS_THIS_MACHINE)
print("Expected total backtests on this machine:", EXPECTED_MAIN_BACKTESTS_THIS_MACHINE + EXPECTED_CONFIRMATION_BACKTESTS_THIS_MACHINE)
print("Expected main Optuna backtests full 4-algo run:", EXPECTED_MAIN_BACKTESTS_FULL)
print("Expected TS confirmation backtests full 4-algo run:", EXPECTED_CONFIRMATION_BACKTESTS_FULL)
print("Expected total backtests full 4-algo run:", EXPECTED_MAIN_BACKTESTS_FULL + EXPECTED_CONFIRMATION_BACKTESTS_FULL)


N_TRIALS per study: 40
Expected main Optuna backtests on this machine: 480
Expected TS confirmation backtests on this machine: 150
Expected total backtests on this machine: 630
Expected main Optuna backtests full 4-algo run: 960
Expected TS confirmation backtests full 4-algo run: 300
Expected total backtests full 4-algo run: 1260


## 3. Загрузка 8 candidates после screening

Файл `selected_feature_sets_for_optuna_balanced.json` должен содержать 2 набора для каждого алгоритма: лучший `corr` и лучший `hybrid`. Notebook дополнительно проверяет, что `mixed` set не попал в Optuna.

In [4]:
if not SELECTED_FOR_OPTUNA_JSON.exists():
    raise FileNotFoundError(f"Не найден файл selected feature sets: {SELECTED_FOR_OPTUNA_JSON}")

with open(SELECTED_FOR_OPTUNA_JSON, "r", encoding="utf-8") as f:
    SELECTED_FOR_OPTUNA = json.load(f)

# Keep only requested algorithms.
missing_algos = [a for a in ALGORITHMS_TO_RUN if a not in SELECTED_FOR_OPTUNA]
if missing_algos:
    raise ValueError(f"В selected JSON нет алгоритмов: {missing_algos}")

FEATURE_PAIRS = []
for algo in ALGORITHMS_TO_RUN:
    items = SELECTED_FOR_OPTUNA[algo]
    if len(items) != 2:
        raise ValueError(f"Ожидалось 2 candidates для {algo}, получено {len(items)}")
    for item in items:
        method_group = item["method_group"]
        set_name = item["set_name"]
        if method_group not in {"corr", "hybrid"}:
            raise ValueError(f"Mixed/unknown method group is not allowed in this notebook: {method_group}")
        if "mixed" in set_name.lower():
            raise ValueError(f"Mixed set is not allowed in this notebook: {set_name}")
        FEATURES = list(item["features"])
        if len(FEATURES) != 10:
            raise ValueError(f"Expected 10 market features for {algo}/{method_group}/{set_name}, got {len(FEATURES)}")
        META = dict(item["meta"])
        FEATURE_PAIRS.append({
            "algorithm": algo,
            "method_group": method_group,
            "set_name": set_name,
            "features": FEATURES,
            "meta": META,
            "z_window": int(META["z_window"]),
            "alpha_out": float(META["alpha_out"]),
        })

feature_pairs_table = pd.DataFrame([
    {
        "algorithm": p["algorithm"],
        "method_group": p["method_group"],
        "set_name": p["set_name"],
        "z_window": p["z_window"],
        "n_market_features": len(p["features"]),
        "n_state_features": len(STATE_FEATURES),
        "n_total_features": len(p["features"]) + len(STATE_FEATURES),
        "features": "|".join(p["features"]),
    }
    for p in FEATURE_PAIRS
])

feature_pairs_table.to_csv(OUTPUT_DIR / "feature_pairs_to_optimize.csv", index=False)
display(feature_pairs_table)

needed_z_windows = sorted(set(feature_pairs_table["z_window"].astype(int)))
print("needed_z_windows:", needed_z_windows)
print("STATE_FEATURES used by Backtesting:", STATE_FEATURES)
print("Expected total context dim for every set:", 10 + len(STATE_FEATURES))


,algorithm,method_group,set_name,z_window,n_market_features,n_state_features,n_total_features,features
0,discounted_lints,corr,z72_a0p5_corr_pruned_top10,72,10,1,11,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...
1,discounted_lints,hybrid,z24_a0p5_hybrid_corr5_stablehie5_top10,24,10,1,11,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...
2,discounted_linucb,corr,z72_a0p5_corr_pruned_top10,72,10,1,11,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...
3,discounted_linucb,hybrid,z24_a0p5_hybrid_corr5_stablehie5_top10,24,10,1,11,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...


needed_z_windows: [24, 72]
STATE_FEATURES used by Backtesting: ['state_in_position']
Expected total context dim for every set: 11


## 4. Data processing helpers

In [5]:
def process_indicators_for_z_window(
    ohlcv_data: pd.DataFrame,
    symbols: list[str],
    meta_cols: list[str],
    indicator_config: dict,
    z_window: int,
    clip_value: float = 5.0,
    shift_by_one: bool = True,
) -> pd.DataFrame:
    processed_parts = []
    for symbol in symbols:
        print(f"Processing {symbol}, z_window={z_window}...")
        df_symbol = (
            ohlcv_data[ohlcv_data["symbol"] == symbol]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )
        indicators_symbol = stream_TA_lib(
            df=df_symbol,
            meta_cols=meta_cols,
            **indicator_config,
        )
        indicators_symbol = indicators_symbol.dropna().sort_values("timestamp").reset_index(drop=True)
        transformed_symbol = transform_indicators_df(
            df=indicators_symbol,
            meta_cols=meta_cols,
        )
        z_score_symbol = rolling_z_score_clip_df(
            df=transformed_symbol,
            meta_cols=meta_cols,
            window=z_window,
            clip_value=clip_value,
            shift_by_one=shift_by_one,
        )
        z_score_symbol = z_score_symbol.dropna().sort_values("timestamp").reset_index(drop=True)
        processed_parts.append(z_score_symbol)

    return (
        pd.concat(processed_parts, ignore_index=True)
        .sort_values(["symbol", "timestamp"])
        .reset_index(drop=True)
    )


def split_train_val_test_by_tail_bars(
    df: pd.DataFrame,
    symbols: list[str],
    val_bars: int,
    test_bars: int,
    embargo_bars: int = 0,
):
    train_parts, val_parts, test_parts, rows = [], [], [], []
    for sym in symbols:
        g = df[df["symbol"] == sym].sort_values("timestamp").reset_index(drop=True).copy()
        n = len(g)
        if n <= val_bars + test_bars + max(embargo_bars, 0):
            raise ValueError(f"Too few rows for {sym}: {n}")

        test_start = n - test_bars
        val_start = test_start - val_bars
        train_end = max(val_start - embargo_bars, 0)
        val_end = test_start - embargo_bars if embargo_bars > 0 else test_start

        train = g.iloc[:train_end].copy()
        val = g.iloc[val_start:val_end].copy()
        test = g.iloc[test_start:].copy()

        train_parts.append(train)
        val_parts.append(val)
        test_parts.append(test)
        rows.append({
            "symbol": sym,
            "n_total": n,
            "n_train": len(train),
            "n_val": len(val),
            "n_test": len(test),
            "train_min": train["timestamp"].min(),
            "train_max": train["timestamp"].max(),
            "val_min": val["timestamp"].min(),
            "val_max": val["timestamp"].max(),
            "test_min": test["timestamp"].min(),
            "test_max": test["timestamp"].max(),
        })

    split_summary = pd.DataFrame(rows)
    train_df = pd.concat(train_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    val_df = pd.concat(val_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    test_df = pd.concat(test_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    return train_df, val_df, test_df, split_summary


def validate_selected_features(train_df: pd.DataFrame, val_df: pd.DataFrame, selected_features: list[str], set_name: str):
    missing_train = [f for f in selected_features if f not in train_df.columns]
    missing_val = [f for f in selected_features if f not in val_df.columns]
    if missing_train or missing_val:
        raise ValueError(f"Missing selected features for {set_name}: train={missing_train}, val={missing_val}")
    values_train = train_df[selected_features].to_numpy(dtype=float)
    values_val = val_df[selected_features].to_numpy(dtype=float)
    if np.isnan(values_train).any() or np.isnan(values_val).any():
        raise ValueError(f"NaN in selected features for {set_name}")
    if np.isinf(values_train).any() or np.isinf(values_val).any():
        raise ValueError(f"inf in selected features for {set_name}")

## 5. Build datasets by z-window

Используется cache в отдельной папке `stage1_optuna_feature_cache`, чтобы не пересчитывать признаки при повторном запуске.

In [6]:
loader = KlinesDataLoader(symbols=ALL_SYMBOLS)
ohlcv_data = loader.load_data(
    download_path=str(OHLCV_RELATIVE_PATH).replace("\\", "/"),
    analyse_data=False,
    cleaning=True,
)
ohlcv_data["timestamp"] = pd.to_datetime(ohlcv_data["timestamp"], utc=True)

DATASETS_BY_Z: dict[int, dict[str, Any]] = {}
split_summaries = []

for z_window in needed_z_windows:
    cache_path = FEATURE_CACHE_DIR / f"datasets_z{z_window}_val{VAL_BARS}_test{TEST_BARS}_embargo{EMBARGO_BARS}.pkl"
    if cache_path.exists():
        print(f"Loading cached dataset for z_window={z_window}: {cache_path}")
        cached = pd.read_pickle(cache_path)
        DATASETS_BY_Z[z_window] = cached["datasets"]
        split_summary = cached["split_summary"].copy()
    else:
        print("=" * 100)
        print(f"Building z_window={z_window}")
        print("=" * 100)
        processed = process_indicators_for_z_window(
            ohlcv_data=ohlcv_data,
            symbols=ALL_SYMBOLS,
            meta_cols=META_COLS,
            indicator_config=CONFIG_FOR_INDICATORS,
            z_window=z_window,
            clip_value=5.0,
            shift_by_one=True,
        )
        train_z, val_z, test_z, split_summary = split_train_val_test_by_tail_bars(
            df=processed,
            symbols=ALL_SYMBOLS,
            val_bars=VAL_BARS,
            test_bars=TEST_BARS,
            embargo_bars=EMBARGO_BARS,
        )
        DATASETS_BY_Z[z_window] = {"train": train_z, "val": val_z, "test": test_z}
        pd.to_pickle({"datasets": DATASETS_BY_Z[z_window], "split_summary": split_summary}, cache_path)

    split_summary = split_summary.copy()
    split_summary.insert(0, "z_window", z_window)
    split_summaries.append(split_summary)

split_summary_used = pd.concat(split_summaries, ignore_index=True)
split_summary_used.to_csv(DIAGNOSTICS_DIR / "split_summary_used.csv", index=False)
display(split_summary_used)

# Validate selected features.
for p in FEATURE_PAIRS:
    z = p["z_window"]
    validate_selected_features(
        train_df=DATASETS_BY_Z[z]["train"],
        val_df=DATASETS_BY_Z[z]["val"],
        selected_features=p["features"],
        set_name=p["set_name"],
    )
print("All selected feature sets validated.")

Building z_window=24
Processing BTCUSDT, z_window=24...
Processing DOGEUSDT, z_window=24...
Processing ETHUSDT, z_window=24...
Processing SOLUSDT, z_window=24...
Processing XRPUSDT, z_window=24...
Building z_window=72
Processing BTCUSDT, z_window=72...
Processing DOGEUSDT, z_window=72...
Processing ETHUSDT, z_window=72...
Processing SOLUSDT, z_window=72...
Processing XRPUSDT, z_window=72...


,z_window,symbol,n_total,n_train,n_val,n_test,train_min,train_max,val_min,val_max,test_min,test_max
0,24,BTCUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
1,24,DOGEUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
2,24,ETHUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
3,24,SOLUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
4,24,XRPUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
5,72,BTCUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
6,72,DOGEUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
7,72,ETHUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
8,72,SOLUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
9,72,XRPUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00


All selected feature sets validated.


## 6. Backtest metrics helpers

In [7]:
def _get_store(backtesting, mode: str, store_name: str):
    if mode == "train":
        return getattr(backtesting, f"{store_name}_train")
    if mode == "val":
        return getattr(backtesting, f"{store_name}_val")
    raise ValueError("mode должен быть 'train' или 'val'")


def extract_trade_stats_from_log(trade_log: list[dict], trade_cost: float) -> pd.DataFrame:
    if not trade_log:
        return pd.DataFrame()
    events = pd.DataFrame(trade_log).copy()
    if events.empty or "event" not in events.columns:
        return pd.DataFrame()
    trades = []
    open_trade = None
    for _, row in events.sort_values("timestamp").iterrows():
        if row["event"] == "BUY":
            open_trade = row
        elif row["event"] == "SELL" and open_trade is not None:
            entry_price = float(open_trade["price"])
            exit_price = float(row["price"])
            pnl_before_cost = np.log(exit_price / entry_price)
            pnl_after_cost = pnl_before_cost + 2.0 * np.log(1.0 - trade_cost)
            trades.append({
                "entry_timestamp": open_trade["timestamp"],
                "exit_timestamp": row["timestamp"],
                "entry_price": entry_price,
                "exit_price": exit_price,
                "pnl_before_cost": pnl_before_cost,
                "pnl_after_cost": pnl_after_cost,
                "holding_time": pd.to_datetime(row["timestamp"]) - pd.to_datetime(open_trade["timestamp"]),
            })
            open_trade = None
    return pd.DataFrame(trades)


def compute_symbol_backtest_metrics(backtesting, sym: str, mode: str, trade_cost: float) -> dict:
    balance_store = _get_store(backtesting, mode, "balance")
    actions_store = _get_store(backtesting, mode, "actions")
    raw_actions_store = _get_store(backtesting, mode, "raw_actions")
    decision_log_store = _get_store(backtesting, mode, "decision_log")
    trade_log_store = _get_store(backtesting, mode, "trade_log")
    reward_log_store = _get_store(backtesting, mode, "reward_log")

    balance = np.asarray(balance_store.get(sym, []), dtype=float)
    actions = np.asarray(actions_store.get(sym, []), dtype=float)
    raw_actions = np.asarray(raw_actions_store.get(sym, []), dtype=float)
    decision_log = pd.DataFrame(decision_log_store.get(sym, []))
    trade_log = trade_log_store.get(sym, [])
    reward_log = pd.DataFrame(reward_log_store.get(sym, []))

    if len(balance) == 0:
        return {"symbol": sym, "mode": mode, "n_decisions": 0}

    start_value = float(balance[0])
    end_value = float(balance[-1])
    return_pct = (end_value / start_value - 1.0) * 100.0
    running_max = np.maximum.accumulate(balance)
    drawdown = balance / (running_max + 1e-12) - 1.0
    max_drawdown_pct = float(drawdown.min() * 100.0)

    rows = {
        "symbol": sym,
        "mode": mode,
        "start_value": start_value,
        "end_value": end_value,
        "return_pct": return_pct,
        "max_drawdown_pct": max_drawdown_pct,
        "drawdown_abs": abs(max_drawdown_pct),
        "raw_action_1_ratio": float(np.mean(raw_actions == 1)) if len(raw_actions) else np.nan,
        "executed_action_1_ratio": float(np.mean(actions == 1)) if len(actions) else np.nan,
        "executed_action_1": int(np.sum(actions == 1)) if len(actions) else 0,
        "raw_action_1": int(np.sum(raw_actions == 1)) if len(raw_actions) else 0,
        "n_decisions": int(len(actions)),
        "trade_events": int(len(trade_log)) if trade_log is not None else 0,
        "turnover": float((len(trade_log) / len(actions)) if len(actions) else np.nan),
    }

    if not decision_log.empty:
        rows.update({
            "threshold_applied_ratio": float(decision_log["threshold_applied"].mean()),
            "constraint_applied_ratio": float(decision_log["constraint_applied"].mean()),
            "abs_edge_mean": float(decision_log["abs_edge"].mean()),
            "abs_edge_p50": float(decision_log["abs_edge"].quantile(0.50)),
            "uncertainty_0_mean": float(decision_log["uncertainty_0"].mean()),
            "uncertainty_1_mean": float(decision_log["uncertainty_1"].mean()),
        })
    else:
        rows.update({
            "threshold_applied_ratio": np.nan,
            "constraint_applied_ratio": np.nan,
            "abs_edge_mean": np.nan,
            "abs_edge_p50": np.nan,
            "uncertainty_0_mean": np.nan,
            "uncertainty_1_mean": np.nan,
        })

    if not reward_log.empty and "reward" in reward_log.columns:
        rew = reward_log["reward"].astype(float)
        rows.update({
            "n_reward_updates": int(len(reward_log)),
            "mean_reward_update": float(rew.mean()),
            "std_reward_update": float(rew.std()),
            "reward_abs_p95": float(rew.abs().quantile(0.95)),
        })
    else:
        rows.update({
            "n_reward_updates": 0,
            "mean_reward_update": np.nan,
            "std_reward_update": np.nan,
            "reward_abs_p95": np.nan,
        })

    trades = extract_trade_stats_from_log(trade_log=trade_log, trade_cost=trade_cost)
    if not trades.empty:
        gross_profit = trades.loc[trades["pnl_after_cost"] > 0, "pnl_after_cost"].sum()
        gross_loss = trades.loc[trades["pnl_after_cost"] < 0, "pnl_after_cost"].sum()
        rows.update({
            "trades": int(len(trades)),
            "win_rate": float((trades["pnl_after_cost"] > 0).mean()),
            "mean_trade_pnl": float(trades["pnl_after_cost"].mean()),
            "median_trade_pnl": float(trades["pnl_after_cost"].median()),
            "total_trade_pnl": float(trades["pnl_after_cost"].sum()),
            "profit_factor": float(gross_profit / abs(gross_loss + 1e-12)),
        })
    else:
        rows.update({
            "trades": 0,
            "win_rate": np.nan,
            "mean_trade_pnl": np.nan,
            "median_trade_pnl": np.nan,
            "total_trade_pnl": 0.0,
            "profit_factor": np.nan,
        })
    return rows


def compute_symbol_backtest_metrics_slice(
    backtesting,
    sym: str,
    source_mode: str,
    period_name: str,
    start_idx: int,
    end_idx: int,
    trade_cost: float,
) -> dict:
    """
    Post-hoc metrics for a contiguous slice of an already executed phase.
    This does not reset or re-run the bandit; it only slices the realized validation trajectory.
    """
    balance_store = _get_store(backtesting, source_mode, "balance")
    actions_store = _get_store(backtesting, source_mode, "actions")
    raw_actions_store = _get_store(backtesting, source_mode, "raw_actions")
    times_store = _get_store(backtesting, source_mode, "times")
    decision_log_store = _get_store(backtesting, source_mode, "decision_log")
    trade_log_store = _get_store(backtesting, source_mode, "trade_log")
    reward_log_store = _get_store(backtesting, source_mode, "reward_log")

    balance_all = np.asarray(balance_store.get(sym, []), dtype=float)
    actions_all = np.asarray(actions_store.get(sym, []), dtype=float)
    raw_actions_all = np.asarray(raw_actions_store.get(sym, []), dtype=float)
    times_all = pd.to_datetime(pd.Series(times_store.get(sym, [])))

    n = len(balance_all)
    start_idx = int(max(0, start_idx))
    end_idx = int(min(n, end_idx))
    if n == 0 or end_idx <= start_idx:
        return {"symbol": sym, "mode": period_name, "n_decisions": 0}

    balance = balance_all[start_idx:end_idx]
    actions = actions_all[start_idx:end_idx]
    raw_actions = raw_actions_all[start_idx:end_idx]
    times = times_all.iloc[start_idx:end_idx].reset_index(drop=True)
    start_ts = times.iloc[0]
    end_ts = times.iloc[-1]

    start_value = float(balance[0])
    end_value = float(balance[-1])
    return_pct = (end_value / start_value - 1.0) * 100.0 if start_value != 0 else np.nan
    running_max = np.maximum.accumulate(balance)
    drawdown = balance / (running_max + 1e-12) - 1.0
    max_drawdown_pct = float(drawdown.min() * 100.0)

    rows = {
        "symbol": sym,
        "mode": period_name,
        "source_mode": source_mode,
        "period_start_idx": start_idx,
        "period_end_idx_exclusive": end_idx,
        "period_start_timestamp": start_ts,
        "period_end_timestamp": end_ts,
        "start_value": start_value,
        "end_value": end_value,
        "return_pct": float(return_pct),
        "max_drawdown_pct": max_drawdown_pct,
        "drawdown_abs": abs(max_drawdown_pct),
        "raw_action_1_ratio": float(np.mean(raw_actions == 1)) if len(raw_actions) else np.nan,
        "executed_action_1_ratio": float(np.mean(actions == 1)) if len(actions) else np.nan,
        "executed_action_1": int(np.sum(actions == 1)) if len(actions) else 0,
        "raw_action_1": int(np.sum(raw_actions == 1)) if len(raw_actions) else 0,
        "n_decisions": int(len(actions)),
    }

    decision_log = pd.DataFrame(decision_log_store.get(sym, []))
    if not decision_log.empty and "timestamp" in decision_log.columns:
        dlog = decision_log.copy()
        dlog["timestamp"] = pd.to_datetime(dlog["timestamp"])
        dlog = dlog[(dlog["timestamp"] >= start_ts) & (dlog["timestamp"] <= end_ts)]
    else:
        dlog = pd.DataFrame()

    if not dlog.empty:
        rows.update({
            "threshold_applied_ratio": float(dlog["threshold_applied"].mean()) if "threshold_applied" in dlog else np.nan,
            "constraint_applied_ratio": float(dlog["constraint_applied"].mean()) if "constraint_applied" in dlog else np.nan,
            "abs_edge_mean": float(dlog["abs_edge"].mean()) if "abs_edge" in dlog else np.nan,
            "abs_edge_p50": float(dlog["abs_edge"].quantile(0.50)) if "abs_edge" in dlog else np.nan,
            "uncertainty_0_mean": float(dlog["uncertainty_0"].mean()) if "uncertainty_0" in dlog else np.nan,
            "uncertainty_1_mean": float(dlog["uncertainty_1"].mean()) if "uncertainty_1" in dlog else np.nan,
        })
    else:
        rows.update({
            "threshold_applied_ratio": np.nan,
            "constraint_applied_ratio": np.nan,
            "abs_edge_mean": np.nan,
            "abs_edge_p50": np.nan,
            "uncertainty_0_mean": np.nan,
            "uncertainty_1_mean": np.nan,
        })

    reward_log = pd.DataFrame(reward_log_store.get(sym, []))
    if not reward_log.empty and "reward" in reward_log.columns:
        rlog = reward_log.copy()
        time_col = "update_timestamp" if "update_timestamp" in rlog.columns else None
        if time_col is not None:
            rlog[time_col] = pd.to_datetime(rlog[time_col])
            rlog = rlog[(rlog[time_col] >= start_ts) & (rlog[time_col] <= end_ts)]
        rew = rlog["reward"].astype(float) if not rlog.empty else pd.Series(dtype=float)
        rows.update({
            "n_reward_updates": int(len(rlog)),
            "mean_reward_update": float(rew.mean()) if len(rew) else np.nan,
            "std_reward_update": float(rew.std()) if len(rew) else np.nan,
            "reward_abs_p95": float(rew.abs().quantile(0.95)) if len(rew) else np.nan,
        })
    else:
        rows.update({
            "n_reward_updates": 0,
            "mean_reward_update": np.nan,
            "std_reward_update": np.nan,
            "reward_abs_p95": np.nan,
        })

    trade_log = trade_log_store.get(sym, [])
    events = pd.DataFrame(trade_log) if trade_log else pd.DataFrame()
    if not events.empty and "timestamp" in events.columns:
        events = events.copy()
        events["timestamp"] = pd.to_datetime(events["timestamp"])
        events_in_period = events[(events["timestamp"] >= start_ts) & (events["timestamp"] <= end_ts)]
        trade_events = int(len(events_in_period))
    else:
        trade_events = 0
    rows["trade_events"] = trade_events
    rows["turnover"] = float(trade_events / len(actions)) if len(actions) else np.nan

    trades = extract_trade_stats_from_log(trade_log=trade_log, trade_cost=trade_cost)
    if not trades.empty and "exit_timestamp" in trades.columns:
        trades = trades.copy()
        trades["exit_timestamp"] = pd.to_datetime(trades["exit_timestamp"])
        trades = trades[(trades["exit_timestamp"] >= start_ts) & (trades["exit_timestamp"] <= end_ts)]

    if not trades.empty:
        gross_profit = trades.loc[trades["pnl_after_cost"] > 0, "pnl_after_cost"].sum()
        gross_loss = trades.loc[trades["pnl_after_cost"] < 0, "pnl_after_cost"].sum()
        rows.update({
            "trades": int(len(trades)),
            "win_rate": float((trades["pnl_after_cost"] > 0).mean()),
            "mean_trade_pnl": float(trades["pnl_after_cost"].mean()),
            "median_trade_pnl": float(trades["pnl_after_cost"].median()),
            "total_trade_pnl": float(trades["pnl_after_cost"].sum()),
            "profit_factor": float(gross_profit / abs(gross_loss + 1e-12)),
        })
    else:
        rows.update({
            "trades": 0,
            "win_rate": np.nan,
            "mean_trade_pnl": np.nan,
            "median_trade_pnl": np.nan,
            "total_trade_pnl": 0.0,
            "profit_factor": np.nan,
        })
    return rows


def summarize_seed_validation(symbol_metrics: pd.DataFrame) -> dict:
    val = symbol_metrics[symbol_metrics["mode"].eq("val")].copy()
    if val.empty:
        raise ValueError("No validation metrics")
    val["profit_factor_clean"] = val["profit_factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    val["is_active"] = val["trades"] > 0
    val["has_long_actions"] = val["executed_action_1"] > 0
    val["is_positive_return"] = val["return_pct"] > 0

    summary = {
        "n_symbols": int(val["symbol"].nunique()),
        "mean_return_pct": float(val["return_pct"].mean()),
        "median_return_pct": float(val["return_pct"].median()),
        "min_return_pct": float(val["return_pct"].min()),
        "max_return_pct": float(val["return_pct"].max()),
        "max_drawdown_abs": float(val["drawdown_abs"].max()),
        "mean_drawdown_abs": float(val["drawdown_abs"].mean()),
        "median_profit_factor": float(val["profit_factor_clean"].median()),
        "mean_profit_factor": float(val["profit_factor_clean"].mean()),
        "total_trades": int(val["trades"].sum()),
        "total_trade_events": int(val["trade_events"].sum()) if "trade_events" in val.columns else int(val["trades"].sum()),
        "mean_trades": float(val["trades"].mean()),
        "mean_turnover": float(val["turnover"].mean()) if "turnover" in val.columns else float(val["trades"].sum() / max(val["n_decisions"].sum(), 1)),
        "mean_win_rate": float(val["win_rate"].mean()),
        "mean_raw_action_1_ratio": float(val["raw_action_1_ratio"].mean()),
        "mean_executed_action_1_ratio": float(val["executed_action_1_ratio"].mean()),
        "mean_threshold_applied_ratio": float(val["threshold_applied_ratio"].mean()),
        "mean_constraint_applied_ratio": float(val["constraint_applied_ratio"].mean()),
        "active_symbols": int(val["is_active"].sum()),
        "long_action_symbols": int(val["has_long_actions"].sum()),
        "positive_return_symbols": int(val["is_positive_return"].sum()),
        "all_symbols_active": bool(val["is_active"].sum() == val["symbol"].nunique()),
        "all_symbols_have_long": bool(val["has_long_actions"].sum() == val["symbol"].nunique()),
    }

    # Optional split-half validation diagnostics. These are not used as the primary objective.
    for period_name in ["val_first", "val_second"]:
        part = symbol_metrics[symbol_metrics["mode"].eq(period_name)].copy()
        if part.empty:
            continue
        part["profit_factor_clean"] = part["profit_factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
        part["is_active"] = part["trades"] > 0
        part["has_long_actions"] = part["executed_action_1"] > 0
        part["is_positive_return"] = part["return_pct"] > 0

        summary.update({
            f"{period_name}_mean_return_pct": float(part["return_pct"].mean()),
            f"{period_name}_median_return_pct": float(part["return_pct"].median()),
            f"{period_name}_min_return_pct": float(part["return_pct"].min()),
            f"{period_name}_max_return_pct": float(part["return_pct"].max()),
            f"{period_name}_max_drawdown_abs": float(part["drawdown_abs"].max()),
            f"{period_name}_mean_drawdown_abs": float(part["drawdown_abs"].mean()),
            f"{period_name}_median_profit_factor": float(part["profit_factor_clean"].median()),
            f"{period_name}_mean_profit_factor": float(part["profit_factor_clean"].mean()),
            f"{period_name}_total_trades": int(part["trades"].sum()),
            f"{period_name}_total_trade_events": int(part["trade_events"].sum()) if "trade_events" in part.columns else int(part["trades"].sum()),
            f"{period_name}_mean_turnover": float(part["turnover"].mean()) if "turnover" in part.columns else np.nan,
            f"{period_name}_mean_executed_action_1_ratio": float(part["executed_action_1_ratio"].mean()),
            f"{period_name}_mean_threshold_applied_ratio": float(part["threshold_applied_ratio"].mean()),
            f"{period_name}_mean_constraint_applied_ratio": float(part["constraint_applied_ratio"].mean()),
            f"{period_name}_positive_return_symbols": int(part["is_positive_return"].sum()),
            f"{period_name}_active_symbols": int(part["is_active"].sum()),
            f"{period_name}_long_action_symbols": int(part["has_long_actions"].sum()),
        })

    first = summary.get("val_first_median_return_pct", np.nan)
    second = summary.get("val_second_median_return_pct", np.nan)
    if np.isfinite(first) and np.isfinite(second):
        summary.update({
            "min_half_median_return_pct": float(min(first, second)),
            "max_half_median_return_pct": float(max(first, second)),
            "half_return_gap_pct": float(abs(second - first)),
            "second_minus_first_half_median_return_pct": float(second - first),
        })
    else:
        summary.update({
            "min_half_median_return_pct": np.nan,
            "max_half_median_return_pct": np.nan,
            "half_return_gap_pct": np.nan,
            "second_minus_first_half_median_return_pct": np.nan,
        })
    return summary


def aggregate_seed_summaries(seed_summaries: list[dict]) -> dict:
    s = pd.DataFrame(seed_summaries)
    out = {
        "n_seeds": int(s["seed"].nunique()),
        "seeds_used": "|".join(str(int(v)) for v in sorted(s["seed"].unique())),
        "all_symbols_active_share": float(s["all_symbols_active"].mean()),
        "all_symbols_have_long_share": float(s["all_symbols_have_long"].mean()),
        "median_of_seed_median_return_pct": float(s["median_return_pct"].median()),
        "mean_of_seed_median_return_pct": float(s["median_return_pct"].mean()),
        "std_of_seed_median_return_pct": float(s["median_return_pct"].std(ddof=1)) if len(s) > 1 else 0.0,
        "min_seed_median_return_pct": float(s["median_return_pct"].min()),
        "max_seed_median_return_pct": float(s["median_return_pct"].max()),
        "median_of_seed_mean_return_pct": float(s["mean_return_pct"].median()),
        "mean_of_seed_mean_return_pct": float(s["mean_return_pct"].mean()),
        "median_max_drawdown_abs": float(s["max_drawdown_abs"].median()),
        "median_profit_factor": float(s["median_profit_factor"].median()),
        "median_total_trades": float(s["total_trades"].median()),
        "median_total_trade_events": float(s["total_trade_events"].median()) if "total_trade_events" in s.columns else float(s["total_trades"].median()),
        "median_mean_turnover": float(s["mean_turnover"].median()) if "mean_turnover" in s.columns else np.nan,
        "median_positive_return_symbols": float(s["positive_return_symbols"].median()),
        "median_active_symbols": float(s["active_symbols"].median()),
        "median_long_action_symbols": float(s["long_action_symbols"].median()),
    }

    # Aggregate split-half diagnostics across TS/UCB seeds.
    split_diag_keys = [
        "val_first_median_return_pct", "val_second_median_return_pct",
        "val_first_mean_return_pct", "val_second_mean_return_pct",
        "val_first_max_drawdown_abs", "val_second_max_drawdown_abs",
        "val_first_median_profit_factor", "val_second_median_profit_factor",
        "val_first_total_trades", "val_second_total_trades",
        "val_first_total_trade_events", "val_second_total_trade_events",
        "val_first_mean_turnover", "val_second_mean_turnover",
        "val_first_mean_executed_action_1_ratio", "val_second_mean_executed_action_1_ratio",
        "val_first_mean_threshold_applied_ratio", "val_second_mean_threshold_applied_ratio",
        "val_first_mean_constraint_applied_ratio", "val_second_mean_constraint_applied_ratio",
        "val_first_positive_return_symbols", "val_second_positive_return_symbols",
        "min_half_median_return_pct", "max_half_median_return_pct",
        "half_return_gap_pct", "second_minus_first_half_median_return_pct",
    ]
    for key in split_diag_keys:
        if key in s.columns:
            ser = pd.to_numeric(s[key], errors="coerce")
            out[f"median_seed_{key}"] = float(ser.median()) if ser.notna().any() else np.nan
            out[f"mean_seed_{key}"] = float(ser.mean()) if ser.notna().any() else np.nan
    return out


## 7. Stage 2 objective: constrained median-of-medians return

Optuna maximizes a robust return-based scalar metric without subjective weights:

$$
\text{objective} = median_{seed}(median_{symbol}(return\_pct))
$$

This is stored as `median_of_seed_median_return_pct`.

### Методологическое обоснование double-median aggregation

В Optuna trial для TS используется небольшое число seeds, поэтому среднее по seeds может быть завышено одной удачной реализацией stochastic exploration. Поэтому основной objective оставлен в той же robust-логике, что и Stage 1: `median_seed(median_symbol(return_pct))`. Внутренний median по symbols описывает типичный результат по 5 активам, а внешний median по seeds снижает влияние seed-level outliers. Mean-based метрики сохраняются как diagnostics. Для consistency со screening основной tie-breaker после double-median objective — `mean_of_seed_median_return_pct`; `mean_of_seed_mean_return_pct` остаётся дополнительной diagnostic-метрикой, но не является primary objective. После Optuna top TS trials дополнительно проверяются на 25 fresh seeds в confirmation stage.


Hard constraints:

- `all_symbols_active_share >= 1.0`;
- `all_symbols_have_long_share >= 1.0`;
- `MIN_TOTAL_TRADES < median_total_trades < MAX_TOTAL_TRADES`;
- `median_profit_factor > MIN_PROFIT_FACTOR`;
- `median_max_drawdown_abs < MAX_DRAWDOWN_ABS`.

Calmar-like ratio is still computed as a diagnostic:

$$
\text{calmar\_like} =
\frac{median_{seed}(median_{symbol}(return\_pct))}
{\max(median_{seed}(max\_drawdown\_abs),\ 1.0)}
$$

The denominator floor `CALMAR_DRAWDOWN_FLOOR_ABS = 1.0` prevents artificially huge diagnostic Calmar values when drawdown is extremely small.

Mean-return metrics, turnover and Calmar-like ratio are diagnostics and tie-breakers only; they are not the optimization target in this version.


**Constraint-aware TPE.** Trials return the real primary objective even when infeasible. Feasibility is passed to `TPESampler(constraints_func=...)` using continuous constraint values where `constraint <= 0` is feasible and `constraint > 0` is a violation. Final ranking and TS confirmation still filter to `valid == True` trials.


In [8]:
def suggest_bandit_params(trial: optuna.Trial, algorithm: str) -> dict:
    params = {
        "bandit_type": BANDIT_TYPE_MAP[algorithm],
        "lambda_prior": trial.suggest_float(
            "lambda_prior",
            SEARCH_SPACE["discounted" if algorithm in DISCOUNTED_BANDITS else "sliding"]["lambda_prior_low"],
            SEARCH_SPACE["discounted" if algorithm in DISCOUNTED_BANDITS else "sliding"]["lambda_prior_high"],
            log=True,
        ),
        "reward_clip": REWARD_CLIP,
    }

    if algorithm in DISCOUNTED_BANDITS:
        one_minus_gamma = trial.suggest_float(
            "one_minus_gamma",
            SEARCH_SPACE["discounted"]["one_minus_gamma_low"],
            SEARCH_SPACE["discounted"]["one_minus_gamma_high"],
            log=True,
        )
        params["discount_factor"] = 1.0 - one_minus_gamma
        trial.set_user_attr("memory_horizon_bars", 1.0 / one_minus_gamma)
        trial.set_user_attr("memory_horizon", 1.0 / one_minus_gamma)
    else:
        params["window_size"] = trial.suggest_int(
            "window_size",
            SEARCH_SPACE["sliding"]["window_size_low"],
            SEARCH_SPACE["sliding"]["window_size_high"],
            step=SEARCH_SPACE["sliding"]["window_size_step"],
        )
        trial.set_user_attr("memory_horizon_bars", float(params["window_size"]))
        trial.set_user_attr("memory_horizon", float(params["window_size"]))

    if algorithm in TS_BANDITS:
        params["noise_std"] = trial.suggest_float(
            "noise_std",
            SEARCH_SPACE["ts"]["noise_std_low"],
            SEARCH_SPACE["ts"]["noise_std_high"],
            log=True,
        )
    else:
        params["ucb_alpha"] = trial.suggest_float(
            "ucb_alpha",
            SEARCH_SPACE["ucb"]["ucb_alpha_low"],
            SEARCH_SPACE["ucb"]["ucb_alpha_high"],
            log=True,
        )
    return params


def default_trial_params(algorithm: str) -> dict:
    """Minimal-state screening defaults enqueued as trial 0 in Stage 2."""
    if algorithm in DISCOUNTED_BANDITS:
        params = {
            "one_minus_gamma": 1.0 / float(SCREENING_DEFAULT_MEMORY_HORIZON_BARS),
            "lambda_prior": float(SCREENING_DEFAULT_LAMBDA_PRIOR),
        }
    else:
        params = {
            "window_size": int(SCREENING_DEFAULT_MEMORY_HORIZON_BARS),
            "lambda_prior": float(SCREENING_DEFAULT_LAMBDA_PRIOR),
        }

    if algorithm in TS_BANDITS:
        params["noise_std"] = float(SCREENING_DEFAULT_NOISE_STD)
    else:
        params["ucb_alpha"] = float(SCREENING_DEFAULT_UCB_ALPHA)

    return params


def compute_memory_horizon_bars(algorithm: str, bandit_params: dict) -> float:
    if algorithm in DISCOUNTED_BANDITS:
        return float(1.0 / (1.0 - float(bandit_params["discount_factor"])))
    return float(bandit_params["window_size"])


def constrained_median_return_objective_from_agg(agg: dict) -> tuple[float, dict, tuple[float, ...]]:
    """
    Stage 2 constrained median-of-medians return objective with Optuna constraints_func support.

    Primary objective returned to Optuna:
        median_of_seed_median_return_pct = median_seed(median_symbol(return_pct))

    Hard feasibility constraints:
        all_symbols_active_share >= 1.0
        all_symbols_have_long_share >= 1.0
        MIN_TOTAL_TRADES < median_total_trades < MAX_TOTAL_TRADES
        median_profit_factor > MIN_PROFIT_FACTOR
        median_max_drawdown_abs < MAX_DRAWDOWN_ABS

    Important Optuna convention:
        constraints_func expects values <= 0 to be feasible and values > 0 to be violations.

    Calmar-like ratio is computed only as a diagnostic, not as the optimization target.
    """
    objective_return = float(agg["median_of_seed_median_return_pct"])
    mean_return = float(agg.get("mean_of_seed_mean_return_pct", np.nan))
    median_drawdown_abs = float(agg["median_max_drawdown_abs"])
    median_profit_factor = float(agg.get("median_profit_factor", np.nan))
    median_total_trades = float(agg["median_total_trades"])
    all_symbols_active_share = float(agg.get("all_symbols_active_share", np.nan))
    all_symbols_have_long_share = float(agg.get("all_symbols_have_long_share", np.nan))

    symbols_active_ok = np.isfinite(all_symbols_active_share) and all_symbols_active_share >= 1.0
    symbols_have_long_ok = np.isfinite(all_symbols_have_long_share) and all_symbols_have_long_share >= 1.0
    min_trades_ok = np.isfinite(median_total_trades) and median_total_trades > float(MIN_TOTAL_TRADES)
    max_trades_ok = np.isfinite(median_total_trades) and median_total_trades < float(MAX_TOTAL_TRADES)
    profit_factor_ok = np.isfinite(median_profit_factor) and median_profit_factor > float(MIN_PROFIT_FACTOR)
    drawdown_ok = np.isfinite(median_drawdown_abs) and median_drawdown_abs > 0 and median_drawdown_abs < float(MAX_DRAWDOWN_ABS)
    finite_objective_return_ok = np.isfinite(objective_return)
    finite_mean_return_ok = np.isfinite(mean_return)

    valid = bool(
        symbols_active_ok
        and symbols_have_long_ok
        and min_trades_ok
        and max_trades_ok
        and profit_factor_ok
        and drawdown_ok
        and finite_objective_return_ok
    )

    effective_drawdown_abs = (
        max(median_drawdown_abs, float(CALMAR_DRAWDOWN_FLOOR_ABS))
        if np.isfinite(median_drawdown_abs) and median_drawdown_abs > 0
        else np.nan
    )
    drawdown_floor_applied = (
        bool(np.isfinite(median_drawdown_abs) and median_drawdown_abs > 0 and median_drawdown_abs < float(CALMAR_DRAWDOWN_FLOOR_ABS))
    )

    calmar_like = (
        objective_return / effective_drawdown_abs
        if finite_objective_return_ok and np.isfinite(effective_drawdown_abs) and effective_drawdown_abs > 0
        else np.nan
    )

    # Optuna constraints_func convention:
    #     value <= 0: feasible
    #     value > 0: violation
    # The magnitude gives TPE information about how far the trial is from feasibility.
    BIG_VIOLATION = 1_000_000.0
    constraint_value_drawdown = (
        median_drawdown_abs - float(MAX_DRAWDOWN_ABS)
        if np.isfinite(median_drawdown_abs)
        else BIG_VIOLATION
    )
    constraint_value_profit_factor = (
        float(MIN_PROFIT_FACTOR) - median_profit_factor
        if np.isfinite(median_profit_factor)
        else BIG_VIOLATION
    )
    constraint_value_min_trades = (
        float(MIN_TOTAL_TRADES) - median_total_trades
        if np.isfinite(median_total_trades)
        else BIG_VIOLATION
    )
    constraint_value_max_trades = (
        median_total_trades - float(MAX_TOTAL_TRADES)
        if np.isfinite(median_total_trades)
        else BIG_VIOLATION
    )
    constraint_value_all_symbols_active = (
        1.0 - all_symbols_active_share
        if np.isfinite(all_symbols_active_share)
        else BIG_VIOLATION
    )
    constraint_value_all_symbols_have_long = (
        1.0 - all_symbols_have_long_share
        if np.isfinite(all_symbols_have_long_share)
        else BIG_VIOLATION
    )
    constraint_value_finite_objective_return = 0.0 if finite_objective_return_ok else BIG_VIOLATION

    constraints_values = (
        float(constraint_value_drawdown),
        float(constraint_value_profit_factor),
        float(constraint_value_min_trades),
        float(constraint_value_max_trades),
        float(constraint_value_all_symbols_active),
        float(constraint_value_all_symbols_have_long),
        float(constraint_value_finite_objective_return),
    )

    # Return the real objective even for infeasible trials.
    # Feasibility is handled by TPESampler.constraints_func and by post-hoc ranking.
    score = float(objective_return) if finite_objective_return_ok else float(INVALID_SCORE)

    details = {
        "valid": valid,
        "constraint_symbols_active_ok": bool(symbols_active_ok),
        "constraint_symbols_have_long_ok": bool(symbols_have_long_ok),
        "constraint_min_trades_ok": bool(min_trades_ok),
        "constraint_max_trades_ok": bool(max_trades_ok),
        "constraint_profit_factor_ok": bool(profit_factor_ok),
        "constraint_drawdown_ok": bool(drawdown_ok),
        "constraint_finite_objective_return_ok": bool(finite_objective_return_ok),
        "objective_metric_name": "median_of_seed_median_return_pct",
        "objective_score": score,
        "calmar_like": float(calmar_like) if np.isfinite(calmar_like) else np.nan,
        "effective_drawdown_abs_for_calmar": float(effective_drawdown_abs) if np.isfinite(effective_drawdown_abs) else np.nan,
        "calmar_drawdown_floor_applied": drawdown_floor_applied,
        "calmar_drawdown_floor_abs": float(CALMAR_DRAWDOWN_FLOOR_ABS),
        "invalid_score": float(INVALID_SCORE),
        "constraints_values": list(constraints_values),
        "constraint_value_drawdown": float(constraint_value_drawdown),
        "constraint_value_profit_factor": float(constraint_value_profit_factor),
        "constraint_value_min_trades": float(constraint_value_min_trades),
        "constraint_value_max_trades": float(constraint_value_max_trades),
        "constraint_value_all_symbols_active": float(constraint_value_all_symbols_active),
        "constraint_value_all_symbols_have_long": float(constraint_value_all_symbols_have_long),
        "constraint_value_finite_objective_return": float(constraint_value_finite_objective_return),
        "optuna_constraints_convention": "feasible_if_all_values_le_0",
        "required_all_symbols_active_share": 1.0,
        "required_all_symbols_have_long_share": 1.0,
        "min_total_trades": float(MIN_TOTAL_TRADES),
        "max_total_trades": float(MAX_TOTAL_TRADES),
        "min_profit_factor": float(MIN_PROFIT_FACTOR),
        "max_drawdown_abs": float(MAX_DRAWDOWN_ABS),
        "observed_all_symbols_active_share": all_symbols_active_share,
        "observed_all_symbols_have_long_share": all_symbols_have_long_share,
        "observed_median_total_trades": median_total_trades,
        "observed_median_profit_factor": median_profit_factor,
        "observed_median_max_drawdown_abs": median_drawdown_abs,
        "observed_median_of_seed_median_return_pct": objective_return,
        "observed_mean_of_seed_mean_return_pct": mean_return,
        # Diagnostics only; not hard constraints in this Stage 2 version.
        "diagnostic_mean_of_seed_mean_return_pct": mean_return,
        "diagnostic_median_mean_turnover": float(agg.get("median_mean_turnover", np.nan)),
    }
    return score, details, constraints_values


def run_backtest_for_seed(
    algorithm: str,
    feature_pair: dict,
    bandit_params: dict,
    seed: int,
) -> tuple[pd.DataFrame, dict]:
    z_window = int(feature_pair["z_window"])
    selected_features = list(feature_pair["features"])
    train_z = DATASETS_BY_Z[z_window]["train"]
    val_z = DATASETS_BY_Z[z_window]["val"]

    config_for_bandit = dict(bandit_params)
    config_for_bandit["n_features"] = len(selected_features) + len(STATE_FEATURES)
    config_for_bandit["actions"] = ACTIONS
    config_for_bandit["seed"] = int(seed)

    bt = Backtesting(
        meta_cols=META_COLS,
        feature_columns=selected_features,
        config_for_bandit=config_for_bandit,
        trade_cost=TRADE_COST,
        seed=int(seed),
        update_on_validation=UPDATE_ON_VALIDATION,
        horizon=HORIZON,
        min_hold_bars=MIN_HOLD_BARS,
        cooldown_bars=COOLDOWN_BARS,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        alpha_out=float(feature_pair["alpha_out"]),
        state_feature_columns=STATE_FEATURES,
    )
    bt.backtest(
        dataframe_train=train_z,
        dataframe_val=val_z,
        symbols=ALL_SYMBOLS,
        start_capital=START_CAPITAL,
        position_size=POSITION_SIZE,
    )

    metric_rows = []
    for sym in ALL_SYMBOLS:
        for mode in ["train", "val"]:
            metric_rows.append(compute_symbol_backtest_metrics(bt, sym, mode, TRADE_COST))

        if SPLIT_VAL_HALVES_FOR_DIAGNOSTICS:
            val_balance = _get_store(bt, "val", "balance").get(sym, [])
            n_val = len(val_balance)
            if n_val >= 2:
                mid = n_val // 2
                metric_rows.append(compute_symbol_backtest_metrics_slice(
                    bt, sym=sym, source_mode="val", period_name=VAL_HALF_MODE_NAMES[0],
                    start_idx=0, end_idx=mid, trade_cost=TRADE_COST,
                ))
                metric_rows.append(compute_symbol_backtest_metrics_slice(
                    bt, sym=sym, source_mode="val", period_name=VAL_HALF_MODE_NAMES[1],
                    start_idx=mid, end_idx=n_val, trade_cost=TRADE_COST,
                ))

    symbol_metrics = pd.DataFrame(metric_rows)
    seed_summary = summarize_seed_validation(symbol_metrics)
    seed_summary["seed"] = int(seed)
    return symbol_metrics, seed_summary


def append_jsonl(path: Path, obj: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=str) + "\n")


## 8. Run Optuna studies

Notebook запускает отдельное study для каждой пары `(algorithm, feature set)`. Storage не указан, поэтому база данных не создаётся. После каждого trial строка результата дописывается в локальный `.jsonl` и `.csv`.

Stage 2 использует return-based objective с hard feasibility constraints. Infeasible trials больше не схлопываются в `INVALID_SCORE = -1_000_000`: реальная return-метрика возвращается в Optuna, а ограничения передаются в `TPESampler(constraints_func=...)`. `enqueue_trial(defaults)` включён: trial 0 в каждой study — это minimal-state screening default baseline (H_mem/window=325, lambda=1, noise_std=0.03 или ucb_alpha=0.10).

Перед full run рекомендуется сначала запустить `SMOKE_TEST = True`. Smoke mode использует малое число trials/seeds и отключает TS confirmation, чтобы быстро проверить, что новые constraints дают хотя бы несколько valid trials. После успешного smoke test нужно поставить `SMOKE_TEST = False` и перезапустить notebook для полного Stage 2 HPO.


In [ ]:
all_trial_rows = []
all_seed_rows = []
all_symbol_rows = []
study_best_rows = []
error_rows = []

# Save global config for reproducibility.
run_config = {
    "created_at": datetime.now().isoformat(),
    "run_label": RUN_LABEL,
    "smoke_test": SMOKE_TEST,
    "enqueue_default_trial": ENQUEUE_DEFAULT_TRIAL,
    "screening_default_trial_params": {
        "memory_horizon_bars": SCREENING_DEFAULT_MEMORY_HORIZON_BARS,
        "lambda_prior": SCREENING_DEFAULT_LAMBDA_PRIOR,
        "noise_std": SCREENING_DEFAULT_NOISE_STD,
        "ucb_alpha": SCREENING_DEFAULT_UCB_ALPHA,
    },
    "algorithms_to_run": ALGORITHMS_TO_RUN,
    "state_features": STATE_FEATURES,
    "n_state_features": len(STATE_FEATURES),
    "expected_total_context_features": 10 + len(STATE_FEATURES),
    "n_trials": N_TRIALS,
    "ts_seeds_per_trial": TS_SEEDS_PER_TRIAL,
    "ucb_seeds_per_trial": UCB_SEEDS_PER_TRIAL,
    "search_space": SEARCH_SPACE,
    "reward_clip_fixed": REWARD_CLIP,
    "objective": {
        "type": "constrained_median_of_seed_median_return",
        "primary_metric": "median_of_seed_median_return_pct",
        "calmar_drawdown_floor_abs": CALMAR_DRAWDOWN_FLOOR_ABS,
        "objective_formula": "median_seed(median_symbol(return_pct))",
        "invalid_score": INVALID_SCORE,
        "constraint_handling": "TPESampler.constraints_func",
        "constraints_convention": "constraint_value <= 0 is feasible; constraint_value > 0 is violation",
        "constraint_values_order": [
            "drawdown",
            "profit_factor",
            "min_total_trades",
            "max_total_trades",
            "all_symbols_active",
            "all_symbols_have_long",
            "finite_objective_return",
        ],
        "min_total_trades": MIN_TOTAL_TRADES,
        "max_total_trades": MAX_TOTAL_TRADES,
        "min_profit_factor": MIN_PROFIT_FACTOR,
        "max_drawdown_abs": MAX_DRAWDOWN_ABS,
        "require_all_symbols_active_share": 1.0,
        "require_all_symbols_have_long_share": 1.0,
        "no_subjective_weights": True,
        "ts_confirmation_stage": RUN_TS_CONFIRMATION,
        "top_n_confirmation": TOP_N_CONFIRMATION,
        "confirmation_seeds": CONFIRMATION_SEEDS,
    },
    "validation_split_diagnostics": {
        "enabled": SPLIT_VAL_HALVES_FOR_DIAGNOSTICS,
        "period_names": list(VAL_HALF_MODE_NAMES),
        "primary_objective_uses_full_val": True,
        "split_half_metrics_are_diagnostics_only": True,
        "split_half_metrics_are_not_constraints": True,
    },
    "execution": {
        "start_capital": START_CAPITAL,
        "position_size": POSITION_SIZE,
        "min_hold_bars": MIN_HOLD_BARS,
        "cooldown_bars": COOLDOWN_BARS,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "update_on_validation": UPDATE_ON_VALIDATION,
        "horizon": HORIZON,
        "trade_cost": TRADE_COST,
    },
    "feature_pairs": feature_pairs_table.to_dict(orient="records"),
}
(OUTPUT_DIR / "optuna_stage2_config.json").write_text(json.dumps(run_config, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

trial_jsonl_path = OUTPUT_DIR / "trial_results.jsonl"
error_jsonl_path = OUTPUT_DIR / "trial_errors.jsonl"


def optuna_constraints_func(frozen_trial):
    """Optuna constraint values: every value <= 0 is feasible; > 0 means violation."""
    values = frozen_trial.user_attrs.get("constraints_values", None)
    if values is None:
        return (1_000_000.0,) * 7
    cleaned = []
    for value in values:
        try:
            value = float(value)
            if not np.isfinite(value):
                value = 1_000_000.0
        except Exception:
            value = 1_000_000.0
        cleaned.append(value)
    if len(cleaned) != 7:
        return (1_000_000.0,) * 7
    return tuple(cleaned)


for pair_idx, feature_pair in enumerate(FEATURE_PAIRS, start=1):
    algorithm = feature_pair["algorithm"]
    method_group = feature_pair["method_group"]
    set_name = feature_pair["set_name"]
    selected_features = feature_pair["features"]
    seeds_for_trial = TS_SEEDS_PER_TRIAL if algorithm in TS_BANDITS else UCB_SEEDS_PER_TRIAL

    study_name = f"{algorithm}__{method_group}__{set_name}"
    safe_study_name = f"{pair_idx:02d}_{algorithm}_{method_group}"
    study_dir = STUDIES_DIR / safe_study_name
    study_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 120)
    print(f"Study {pair_idx}/{len(FEATURE_PAIRS)}: {study_name}")
    print("features:", selected_features)
    print("seeds:", seeds_for_trial)
    print("=" * 120)

    sampler = optuna.samplers.TPESampler(
        seed=OPTUNA_SAMPLER_SEED + pair_idx,
        n_startup_trials=min(10, max(1, N_TRIALS // 4)),
        multivariate=True,
        group=True,
        constraints_func=optuna_constraints_func,
    )
    study = optuna.create_study(
        study_name=study_name,
        direction="maximize",
        sampler=sampler,
        pruner=optuna.pruners.NopPruner(),
        storage=None,
    )

    # Enqueue the minimal-state screening default as trial 0.
    # This gives every study a known baseline from the screening stage.
    if ENQUEUE_DEFAULT_TRIAL:
        default_params = default_trial_params(algorithm)
        print("Enqueue screening default trial:", default_params)
        study.enqueue_trial(default_params)

    def objective(trial: optuna.Trial) -> float:
        bandit_params = suggest_bandit_params(trial, algorithm)
        memory_horizon_bars_val = compute_memory_horizon_bars(algorithm, bandit_params)
        seed_summaries = []
        symbol_metric_frames = []
        try:
            for seed in seeds_for_trial:
                symbol_metrics, seed_summary = run_backtest_for_seed(
                    algorithm=algorithm,
                    feature_pair=feature_pair,
                    bandit_params=bandit_params,
                    seed=int(seed),
                )
                symbol_metrics.insert(0, "seed", int(seed))
                symbol_metrics.insert(0, "trial_number", int(trial.number))
                symbol_metrics.insert(0, "set_name", set_name)
                symbol_metrics.insert(0, "method_group", method_group)
                symbol_metrics.insert(0, "bandit_name", algorithm)
                symbol_metric_frames.append(symbol_metrics)

                seed_summary.update({
                    "trial_number": int(trial.number),
                    "bandit_name": algorithm,
                    "method_group": method_group,
                    "set_name": set_name,
                    "z_window": feature_pair["z_window"],
                })
                seed_summaries.append(seed_summary)

            agg = aggregate_seed_summaries(seed_summaries)
            score, constraint_details, constraints_values = constrained_median_return_objective_from_agg(agg)

            trial_row = {
                "trial_number": int(trial.number),
                "study_name": study_name,
                "bandit_name": algorithm,
                "method_group": method_group,
                "set_name": set_name,
                "z_window": feature_pair["z_window"],
                "alpha_out": feature_pair["alpha_out"],
                "n_market_features": len(selected_features),
                "feature_list": "|".join(selected_features),
                "memory_horizon_bars": float(memory_horizon_bars_val),
                "objective_score": score,
                **constraint_details,
                **agg,
                **{f"param_{k}": v for k, v in bandit_params.items() if k not in {"actions"}},
                **{f"optuna_param_{k}": v for k, v in trial.params.items()},
            }

            all_trial_rows.append(trial_row)
            all_seed_rows.extend(seed_summaries)
            if symbol_metric_frames:
                all_symbol_rows.append(pd.concat(symbol_metric_frames, ignore_index=True))

            append_jsonl(trial_jsonl_path, trial_row)
            pd.DataFrame(all_trial_rows).to_csv(OUTPUT_DIR / "trial_results_all.csv", index=False)
            pd.DataFrame(all_seed_rows).to_csv(OUTPUT_DIR / "trial_seed_level_summary_all.csv", index=False)
            if all_symbol_rows:
                pd.concat(all_symbol_rows, ignore_index=True).to_csv(DIAGNOSTICS_DIR / "trial_symbol_metrics_all.csv", index=False)

            trial.set_user_attr("objective_score", score)
            trial.set_user_attr("memory_horizon_bars", float(memory_horizon_bars_val))
            trial.set_user_attr("constraints_values", list(constraints_values))
            for k, v in constraint_details.items():
                trial.set_user_attr(k, v)
            for k, v in agg.items():
                trial.set_user_attr(k, v)
            return score

        except Exception as e:
            err = {
                "trial_number": int(trial.number),
                "study_name": study_name,
                "bandit_name": algorithm,
                "method_group": method_group,
                "set_name": set_name,
                "error": repr(e),
                "params": dict(trial.params),
            }
            error_rows.append(err)
            append_jsonl(error_jsonl_path, err)
            pd.DataFrame(error_rows).to_csv(OUTPUT_DIR / "trial_errors.csv", index=False)
            print("TRIAL ERROR:", err)
            return -1e12

    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, gc_after_trial=True, show_progress_bar=True)

    # Save Optuna's native trial table for this study.
    study_trials_df = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
    study_dir.mkdir(parents=True, exist_ok=True)
    study_trials_df.to_csv(study_dir / "optuna_trials_dataframe.csv", index=False)

    # Feasibility diagnostics: if all trials are invalid, constraints are too strict for this study.
    valid_rate = np.nan
    feasibility_failed = False
    try:
        valid_rate = float(study_trials_df["user_attrs_valid"].astype(str).str.lower().eq("true").mean())
        feasibility_failed = bool(valid_rate == 0.0)
        print(f"Valid trial rate for {study_name}: {valid_rate:.2%}")
        if feasibility_failed:
            print("WARNING: 0 valid trials. Consider relaxing MIN_TOTAL_TRADES / MAX_TOTAL_TRADES or running more trials.")
    except Exception as e:
        print("Could not compute valid trial rate:", e)

    # With TPESampler.constraints_func, study.best_trial is the best FEASIBLE trial.
    # If there are no feasible completed trials, Optuna raises ValueError. Keep the notebook
    # running and fall back to the raw objective best only for diagnostics/artifacts.
    completed_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None and np.isfinite(float(t.value))
    ]
    try:
        raw_best = study.best_trial
        raw_best_source = "optuna_best_feasible"
    except ValueError:
        if completed_trials:
            raw_best = max(completed_trials, key=lambda t: float(t.value))
            raw_best_source = "raw_objective_best_no_feasible_trials"
            print(f"WARNING: study.best_trial unavailable for {study_name}: no feasible completed trials. "
                  f"Using raw objective best trial {raw_best.number} only as fallback artifact.")
        else:
            raise RuntimeError(f"No completed trials for study {study_name}; cannot build study summary.")

    # For artifacts used by the project, save the best FEASIBLE trial according to the same post-hoc ranking
    # used later in final_trial_ranking_all.csv. If no feasible trial exists, fall back to raw objective best.
    current_study_trials_df = pd.DataFrame([r for r in all_trial_rows if r.get("study_name") == study_name])
    selected_source = "raw_optuna_best_fallback"
    if not current_study_trials_df.empty:
        summary_ranking_cols = [
            "valid",
            "objective_score",
            "median_of_seed_median_return_pct",
            "mean_of_seed_median_return_pct",
            "median_seed_min_half_median_return_pct",
            "median_seed_half_return_gap_pct",
            "median_max_drawdown_abs",
            "median_profit_factor",
            "median_total_trades",
            "memory_horizon_bars",
        ]
        available_summary_cols = [c for c in summary_ranking_cols if c in current_study_trials_df.columns]
        ascending_map = {
            "valid": False,
            "objective_score": False,
            "median_of_seed_median_return_pct": False,
            "mean_of_seed_median_return_pct": False,
            "median_seed_min_half_median_return_pct": False,
            "median_seed_half_return_gap_pct": True,
            "median_max_drawdown_abs": True,
            "median_profit_factor": False,
            "median_total_trades": True,
            "memory_horizon_bars": True,
        }
        study_ranked_for_summary = current_study_trials_df.sort_values(
            available_summary_cols,
            ascending=[ascending_map[c] for c in available_summary_cols],
        )
        feasible_for_summary = study_ranked_for_summary[study_ranked_for_summary["valid"].astype(bool)].copy()
        if not feasible_for_summary.empty:
            selected_summary_row = feasible_for_summary.iloc[0].to_dict()
            selected_source = "best_feasible_posthoc_ranking"
        else:
            selected_summary_row = study_ranked_for_summary.iloc[0].to_dict()
            selected_source = "raw_objective_best_no_feasible_trials"
    else:
        selected_summary_row = {
            "trial_number": int(raw_best.number),
            "objective_score": float(raw_best.value),
        }

    best_params_from_selected = {
        c.replace("optuna_param_", ""): selected_summary_row[c]
        for c in selected_summary_row
        if str(c).startswith("optuna_param_") and pd.notna(selected_summary_row[c])
    }

    feasibility_failed = bool(feasibility_failed)
    best_row = {
        "study_name": study_name,
        "bandit_name": algorithm,
        "method_group": method_group,
        "set_name": set_name,
        "best_trial_number": int(selected_summary_row.get("trial_number", raw_best.number)),
        "best_objective_score": float(selected_summary_row.get("objective_score", raw_best.value)),
        "best_selection_source": selected_source,
        "raw_optuna_best_trial_number": int(raw_best.number),
        "raw_optuna_best_objective_score": float(raw_best.value),
        "raw_optuna_best_valid": bool(raw_best.user_attrs.get("valid", False)),
        "raw_optuna_best_source": raw_best_source,
        "study_valid_trial_rate": float(valid_rate) if np.isfinite(valid_rate) else np.nan,
        "study_feasibility_failed": feasibility_failed,
        **{f"best_param_{k}": v for k, v in best_params_from_selected.items()},
        **{f"best_metric_{k}": v for k, v in selected_summary_row.items() if k in {
            "valid",
            "median_of_seed_median_return_pct",
            "mean_of_seed_median_return_pct",
            "median_seed_min_half_median_return_pct",
            "median_seed_half_return_gap_pct",
            "median_max_drawdown_abs",
            "median_profit_factor",
            "median_total_trades",
            "calmar_like",
            "memory_horizon_bars",
        }},
        **{f"raw_best_user_{k}": v for k, v in raw_best.user_attrs.items()},
    }
    study_best_rows.append(best_row)
    pd.DataFrame(study_best_rows).to_csv(OUTPUT_DIR / "study_best_summary.csv", index=False)

    with open(study_dir / "best_trial.json", "w", encoding="utf-8") as f:
        json.dump(best_row, f, ensure_ascii=False, indent=2, default=str)

    gc.collect()

print("Done.")
print("OUTPUT_DIR:", OUTPUT_DIR)


C:\Users\trosh\AppData\Local\Temp\ipykernel_11100\2379341141.py:118: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
C:\Users\trosh\AppData\Local\Temp\ipykernel_11100\2379341141.py:118: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
C:\Users\trosh\AppData\Local\Temp\ipykernel_11100\2379341141.py:118: ExperimentalWarning: Argument ``constraints_func`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
[I 2026-05-18 19:35:42,911] A new study created in memory with name: discounted_lints__corr__z72_a0p5_corr_pruned_top10


Study 1/4: discounted_lints__corr__z72_a0p5_corr_pruned_top10
features: ['range_sqrt_z', 'vol_ratio_72_log1p_z', 'dist_to_high_168_sqrt_z', 'body_sqrt_z', 'bb_width_20_sqrt_z', 'NATR_14_z', 'ADX_14_scaled', 'MOM_pct_14_z', 'MACD_hist_pct_z', 'volatility_24_sqrt_z']
seeds: [3142, 3143, 3144, 3145, 3146]
Enqueue screening default trial: {'one_minus_gamma': 0.003076923076923077, 'lambda_prior': 1.0, 'noise_std': 0.03}


  0%|          | 0/40 [00:00<?, ?it/s]

BTCUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
BTCUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
BTCUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
BTCUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
DOGEUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
DOGEUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
DOGEUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
DOGEUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
ETHUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
ETHUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
ETHUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
ETHUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
SOLUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
SOLUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
SOLUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
SOLUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
XRPUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
XRPUSDT: фаза train закончилась: 202

  2%|▎         | 1/40 [05:13<3:23:35, 313.23s/it]

...

[I 2026-05-18 19:46:00,547] Trial 1 finished with value: -0.8995699331185381 and parameters: {'lambda_prior': 2.6583217675071293, 'one_minus_gamma': 0.0029684731211750454, 'noise_std': 0.024053363624212343}. Best is trial 1 with value: -0.8995699331185381.
BTCUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
BTCUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
BTCUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
BTCUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
DOGEUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
DOGEUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
DOGEUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
DOGEUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
ETHUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
ETHUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
ETHUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
ETHUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
SOLUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
SOLUS

Best trial: 1. Best value: -0.89957:   8%|▊         | 3/40 [15:24<3:09:33, 307.40s/it]

[I 2026-05-18 19:51:07,141] Trial 2 finished with value: -1.1310760124910613 and parameters: {'lambda_prior': 0.45417725253359703, 'one_minus_gamma': 0.003004412480246536, 'noise_std': 0.0138940336513385}. Best is trial 1 with value: -0.8995699331185381.
BTCUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
BTCUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
BTCUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
BTCUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
DOGEUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
DOGEUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
DOGEUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
DOGEUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
ETHUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
ETHUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
ETHUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
ETHUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
SOLUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
SOLUSDT

Best trial: 1. Best value: -0.89957:  10%|█         | 4/40 [20:11<2:59:43, 299.54s/it]

[I 2026-05-18 19:55:54,642] Trial 3 finished with value: -3.3263063365936874 and parameters: {'lambda_prior': 0.4103210507454176, 'one_minus_gamma': 0.0036919324365728744, 'noise_std': 0.028224331063458036}. Best is trial 1 with value: -0.8995699331185381.
BTCUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
BTCUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
BTCUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
BTCUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
DOGEUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
DOGEUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
DOGEUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
DOGEUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
ETHUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
ETHUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
ETHUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
ETHUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
SOLUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
SOLUS

## 9. Final ranking for this machine

Сортировка после Optuna:

1. valid trials first;
2. higher objective score (`median_of_seed_median_return_pct`);
3. higher `mean_of_seed_median_return_pct` as a secondary diagnostic, matching the screening ranking logic;
4. lower drawdown;
5. higher profit factor;
6. fewer trades as tie-breaker, because Stage 2 is designed to avoid overtrading.

Calmar-like ratio remains in the output as a risk-adjusted diagnostic, but it is not the optimization target.


In [ ]:
trial_results = pd.read_csv(OUTPUT_DIR / "trial_results_all.csv")

# Main selection remains full-validation return based.
# Split-half diagnostics are used only as tie-breakers after the main return/risk/trade criteria.
ranking_cols = [
    "valid",
    "objective_score",
    "median_of_seed_median_return_pct",
    "mean_of_seed_median_return_pct",
    "calmar_like",
    "median_max_drawdown_abs",
    "median_profit_factor",
    "median_total_trades",
    "median_seed_min_half_median_return_pct",
    "median_seed_half_return_gap_pct",
    "memory_horizon_bars",
]

# Higher is better for valid/objective/returns/calmar/PF/min-half; lower is better for DD/trades/gap/memory.
ranking_ascending = [False, False, False, False, False, True, False, True, False, True, True]

optional_split_diag_cols = [
    "median_seed_min_half_median_return_pct",
    "median_seed_half_return_gap_pct",
    "median_seed_val_first_median_return_pct",
    "median_seed_val_second_median_return_pct",
]
for col in optional_split_diag_cols:
    if col not in trial_results.columns:
        trial_results[col] = np.nan

missing_ranking_cols = [c for c in ranking_cols if c not in trial_results.columns]
if missing_ranking_cols:
    raise KeyError(f"Missing ranking columns in trial_results_all.csv: {missing_ranking_cols}")

final_ranking = (
    trial_results
    .sort_values(
        ["bandit_name", "method_group", *ranking_cols],
        ascending=[True, True, *ranking_ascending],
    )
    .reset_index(drop=True)
)
final_ranking["rank_within_algorithm_method"] = final_ranking.groupby(["bandit_name", "method_group"]).cumcount() + 1

# Best trial for every algorithm × method group under the main full-val selection rule.
best_per_pair = final_ranking[final_ranking["rank_within_algorithm_method"].eq(1)].copy()

# Best method comparison within each algorithm under the same main rule.
best_method_comparison = (
    best_per_pair
    .sort_values(
        ["bandit_name", *ranking_cols],
        ascending=[True, *ranking_ascending],
    )
    .reset_index(drop=True)
)
best_method_comparison["rank_within_algorithm"] = best_method_comparison.groupby("bandit_name").cumcount() + 1

# Alternative diagnostic ranking: prioritizes the better worst-half validation return.
# This file is for analysis only; TS confirmation below still uses best_per_pair from the main full-val ranking.
half_robust_ranking_cols = [
    "valid",
    "median_seed_min_half_median_return_pct",
    "objective_score",
    "median_of_seed_median_return_pct",
    "median_seed_half_return_gap_pct",
    "calmar_like",
    "median_max_drawdown_abs",
    "median_profit_factor",
    "median_total_trades",
    "memory_horizon_bars",
]
half_robust_ascending = [False, False, False, False, True, False, True, False, True, True]
half_robust_ranking = (
    trial_results
    .sort_values(
        ["bandit_name", "method_group", *half_robust_ranking_cols],
        ascending=[True, True, *half_robust_ascending],
    )
    .reset_index(drop=True)
)
half_robust_ranking["half_robust_rank_within_algorithm_method"] = (
    half_robust_ranking.groupby(["bandit_name", "method_group"]).cumcount() + 1
)
half_robust_best_per_pair = half_robust_ranking[
    half_robust_ranking["half_robust_rank_within_algorithm_method"].eq(1)
].copy()

final_ranking.to_csv(OUTPUT_DIR / "final_trial_ranking_all.csv", index=False)
best_per_pair.to_csv(OUTPUT_DIR / "best_trial_per_algorithm_method.csv", index=False)
best_method_comparison.to_csv(OUTPUT_DIR / "best_method_comparison_within_algorithm.csv", index=False)
half_robust_ranking.to_csv(OUTPUT_DIR / "diagnostic_half_robust_trial_ranking_all.csv", index=False)
half_robust_best_per_pair.to_csv(OUTPUT_DIR / "diagnostic_half_robust_best_trial_per_algorithm_method.csv", index=False)

display_cols = [
    "bandit_name", "method_group", "set_name", "trial_number", "valid",
    "median_of_seed_median_return_pct", "mean_of_seed_median_return_pct",
    "median_seed_val_first_median_return_pct", "median_seed_val_second_median_return_pct",
    "median_seed_min_half_median_return_pct", "median_seed_half_return_gap_pct",
    "median_max_drawdown_abs", "median_profit_factor", "median_total_trades", "median_mean_turnover",
    "calmar_like", "memory_horizon_bars", "objective_score",
]

display(best_per_pair[[c for c in display_cols if c in best_per_pair.columns]])

display(best_method_comparison[[
    c for c in ["bandit_name", "method_group", "set_name", "trial_number", "rank_within_algorithm", "valid", *display_cols[5:]]
    if c in best_method_comparison.columns
]])

print("Diagnostic half-robust winners. These are NOT used automatically for TS confirmation/final selection.")
display(half_robust_best_per_pair[[c for c in display_cols if c in half_robust_best_per_pair.columns]])


## 10. TS confirmation: top-3 trials × 25 fresh seeds

Для Thompson Sampling (`discounted_lints`, `sw_lints`) после основного Optuna top-3 trials в каждой study переоцениваются на `25` fresh seeds.

Это **не новая оптимизация**: параметры фиксируются, новые параметры не ищутся. Confirmation нужен только для проверки seed-robustness top trials.

Для UCB confirmation не запускается, потому что UCB в текущей реализации детерминированный.


In [ ]:
confirmation_rows = []
confirmation_seed_rows = []
confirmation_symbol_frames = []

# Lookup to recover feature_pair metadata by algorithm/method/set.
feature_pair_lookup = {
    (fp["algorithm"], fp["method_group"], fp["set_name"]): fp
    for fp in FEATURE_PAIRS
}


def bandit_params_from_trial_result_row(row: pd.Series) -> dict:
    """Restore executable bandit params from saved trial_results row."""
    algorithm = row["bandit_name"]
    params = {
        "bandit_type": BANDIT_TYPE_MAP[algorithm],
        "lambda_prior": float(row["param_lambda_prior"]),
        "reward_clip": REWARD_CLIP,
    }
    if algorithm in DISCOUNTED_BANDITS:
        params["discount_factor"] = float(row["param_discount_factor"])
    else:
        params["window_size"] = int(row["param_window_size"])

    if algorithm in TS_BANDITS:
        params["noise_std"] = float(row["param_noise_std"])
    else:
        params["ucb_alpha"] = float(row["param_ucb_alpha"])
    return params


if RUN_TS_CONFIRMATION:
    # Confirm only valid TS trials. Invalid trials have INVALID_SCORE and should not consume confirmation compute.
    ts_ranking = final_ranking[
        final_ranking["bandit_name"].isin(TS_BANDITS)
        & final_ranking["valid"].astype(bool)
    ].copy()

    if ts_ranking.empty:
        print("No valid TS studies in this RUN_PRESET; confirmation skipped.")
    else:
        for (algorithm, method_group, set_name), part in ts_ranking.groupby(["bandit_name", "method_group", "set_name"], sort=True):
            top_trials = part.sort_values(
                ranking_cols,
                ascending=ranking_ascending,
            ).head(TOP_N_CONFIRMATION).copy()

            feature_pair = feature_pair_lookup[(algorithm, method_group, set_name)]

            for confirmation_rank, (_, trial_row) in enumerate(top_trials.iterrows(), start=1):
                original_trial_number = int(trial_row["trial_number"])
                original_objective = float(trial_row["objective_score"])
                bandit_params = bandit_params_from_trial_result_row(trial_row)

                print("=" * 120)
                print(
                    f"TS confirmation: {algorithm} | {method_group} | {set_name} | "
                    f"original trial={original_trial_number} | rank={confirmation_rank}/{TOP_N_CONFIRMATION}"
                )
                print("params:", bandit_params)
                print("confirmation seeds:", CONFIRMATION_SEEDS)
                print("=" * 120)

                seed_summaries = []
                symbol_metric_frames = []

                for seed in CONFIRMATION_SEEDS:
                    symbol_metrics, seed_summary = run_backtest_for_seed(
                        algorithm=algorithm,
                        feature_pair=feature_pair,
                        bandit_params=bandit_params,
                        seed=int(seed),
                    )
                    symbol_metrics.insert(0, "confirmation_seed", int(seed))
                    symbol_metrics.insert(0, "original_trial_number", original_trial_number)
                    symbol_metrics.insert(0, "confirmation_rank_from_optuna", confirmation_rank)
                    symbol_metrics.insert(0, "set_name", set_name)
                    symbol_metrics.insert(0, "method_group", method_group)
                    symbol_metrics.insert(0, "bandit_name", algorithm)
                    symbol_metric_frames.append(symbol_metrics)

                    seed_summary.update({
                        "confirmation_seed": int(seed),
                        "original_trial_number": original_trial_number,
                        "confirmation_rank_from_optuna": confirmation_rank,
                        "bandit_name": algorithm,
                        "method_group": method_group,
                        "set_name": set_name,
                        "z_window": feature_pair["z_window"],
                    })
                    seed_summaries.append(seed_summary)

                agg = aggregate_seed_summaries(seed_summaries)
                confirmation_score, constraint_details, constraints_values = constrained_median_return_objective_from_agg(agg)

                row = {
                    "bandit_name": algorithm,
                    "method_group": method_group,
                    "set_name": set_name,
                    "z_window": feature_pair["z_window"],
                    "alpha_out": feature_pair["alpha_out"],
                    "original_trial_number": original_trial_number,
                    "confirmation_rank_from_optuna": confirmation_rank,
                    "original_objective_score": original_objective,
                    "confirmation_objective_score": confirmation_score,
                    "memory_horizon_bars": float(1.0 / (1.0 - float(bandit_params["discount_factor"])) if algorithm in DISCOUNTED_BANDITS else int(bandit_params["window_size"])),
                    "n_confirmation_seeds": len(CONFIRMATION_SEEDS),
                    "confirmation_seeds": "|".join(str(s) for s in CONFIRMATION_SEEDS),
                    **constraint_details,
                    **{f"confirmation_{k}": v for k, v in agg.items()},
                    **{f"param_{k}": v for k, v in bandit_params.items() if k not in {"actions"}},
                }
                confirmation_rows.append(row)
                confirmation_seed_rows.extend(seed_summaries)
                if symbol_metric_frames:
                    confirmation_symbol_frames.append(pd.concat(symbol_metric_frames, ignore_index=True))

                pd.DataFrame(confirmation_rows).to_csv(OUTPUT_DIR / "ts_top3_confirmation_results.csv", index=False)
                pd.DataFrame(confirmation_seed_rows).to_csv(OUTPUT_DIR / "ts_top3_confirmation_seed_level.csv", index=False)
                if confirmation_symbol_frames:
                    pd.concat(confirmation_symbol_frames, ignore_index=True).to_csv(
                        DIAGNOSTICS_DIR / "ts_top3_confirmation_symbol_metrics.csv",
                        index=False,
                    )

    if confirmation_rows:
        confirmation_df = pd.DataFrame(confirmation_rows)
        confirmation_ranking = (
            confirmation_df
            .sort_values(
                [
                    "bandit_name", "method_group",
                    "valid",
                    "confirmation_objective_score",
                    "confirmation_median_of_seed_median_return_pct",
                    "confirmation_mean_of_seed_median_return_pct",
                    "confirmation_median_seed_min_half_median_return_pct",
                    "confirmation_median_seed_half_return_gap_pct",
                    "confirmation_median_max_drawdown_abs",
                    "confirmation_median_profit_factor",
                    "confirmation_median_total_trades",
                    "memory_horizon_bars",
                ],
                ascending=[True, True, False, False, False, False, False, True, True, False, True, True],
            )
            .reset_index(drop=True)
        )
        confirmation_ranking["confirmation_rank_within_algorithm_method"] = (
            confirmation_ranking.groupby(["bandit_name", "method_group"]).cumcount() + 1
        )
        confirmation_best_per_pair = confirmation_ranking[
            confirmation_ranking["confirmation_rank_within_algorithm_method"].eq(1)
        ].copy()

        confirmation_ranking.to_csv(OUTPUT_DIR / "ts_top3_confirmation_ranking.csv", index=False)
        confirmation_best_per_pair.to_csv(OUTPUT_DIR / "ts_best_trial_per_algorithm_method_after_confirmation.csv", index=False)

        # Unified final selection table: TS uses confirmation winner; UCB uses Optuna winner.
        final_selection_rows = []
        for _, row in best_per_pair.iterrows():
            algorithm = row["bandit_name"]
            method_group = row["method_group"]
            if algorithm in TS_BANDITS:
                match = confirmation_best_per_pair[
                    (confirmation_best_per_pair["bandit_name"].eq(algorithm))
                    & (confirmation_best_per_pair["method_group"].eq(method_group))
                ]
                if not match.empty:
                    conf = match.iloc[0]
                    final_selection_rows.append({
                        "selection_source": "ts_top3_25seed_confirmation",
                        "bandit_name": algorithm,
                        "method_group": method_group,
                        "set_name": conf["set_name"],
                        "selected_trial_number": int(conf["original_trial_number"]),
                        "selected_objective_score": float(conf["confirmation_objective_score"]),
                        "selected_valid": bool(conf["valid"]),
                        "selected_mean_of_seed_mean_return_pct": float(conf.get("confirmation_mean_of_seed_mean_return_pct", np.nan)),
                        "selected_median_of_seed_median_return_pct": float(conf["confirmation_median_of_seed_median_return_pct"]),
                        "selected_mean_of_seed_median_return_pct": float(conf["confirmation_mean_of_seed_median_return_pct"]),
                        "selected_std_of_seed_median_return_pct": float(conf["confirmation_std_of_seed_median_return_pct"]),
                        "selected_median_max_drawdown_abs": float(conf["confirmation_median_max_drawdown_abs"]),
                        "selected_median_profit_factor": float(conf["confirmation_median_profit_factor"]),
                        "selected_median_total_trades": float(conf["confirmation_median_total_trades"]),
                        "selected_median_seed_val_first_median_return_pct": float(conf.get("confirmation_median_seed_val_first_median_return_pct", np.nan)),
                        "selected_median_seed_val_second_median_return_pct": float(conf.get("confirmation_median_seed_val_second_median_return_pct", np.nan)),
                        "selected_median_seed_min_half_median_return_pct": float(conf.get("confirmation_median_seed_min_half_median_return_pct", np.nan)),
                        "selected_median_seed_half_return_gap_pct": float(conf.get("confirmation_median_seed_half_return_gap_pct", np.nan)),
                        "selected_memory_horizon_bars": float(conf.get("memory_horizon_bars", np.nan)),
                        "n_selection_seeds": int(conf["n_confirmation_seeds"]),
                        "original_optuna_objective_score": float(conf["original_objective_score"]),
                    })
                    continue

            # UCB or fallback if confirmation missing.
            final_selection_rows.append({
                "selection_source": "optuna_5seed_or_ucb",
                "bandit_name": algorithm,
                "method_group": method_group,
                "set_name": row["set_name"],
                "selected_trial_number": int(row["trial_number"]),
                "selected_objective_score": float(row["objective_score"]),
                "selected_valid": bool(row["valid"]),
                "selected_mean_of_seed_mean_return_pct": float(row.get("mean_of_seed_mean_return_pct", np.nan)),
                "selected_median_of_seed_median_return_pct": float(row["median_of_seed_median_return_pct"]),
                "selected_mean_of_seed_median_return_pct": float(row["mean_of_seed_median_return_pct"]),
                "selected_std_of_seed_median_return_pct": float(row.get("std_of_seed_median_return_pct", 0.0)),
                "selected_median_max_drawdown_abs": float(row["median_max_drawdown_abs"]),
                "selected_median_profit_factor": float(row["median_profit_factor"]),
                "selected_median_total_trades": float(row["median_total_trades"]),
                "selected_median_seed_val_first_median_return_pct": float(row.get("median_seed_val_first_median_return_pct", np.nan)),
                "selected_median_seed_val_second_median_return_pct": float(row.get("median_seed_val_second_median_return_pct", np.nan)),
                "selected_median_seed_min_half_median_return_pct": float(row.get("median_seed_min_half_median_return_pct", np.nan)),
                "selected_median_seed_half_return_gap_pct": float(row.get("median_seed_half_return_gap_pct", np.nan)),
                "selected_memory_horizon_bars": float(row.get("memory_horizon_bars", np.nan)),
                "n_selection_seeds": int(row["n_seeds"]),
                "original_optuna_objective_score": float(row["objective_score"]),
            })

        final_selection = pd.DataFrame(final_selection_rows)
        final_selection.to_csv(OUTPUT_DIR / "best_trial_per_algorithm_method_with_ts_confirmation.csv", index=False)

        display(confirmation_best_per_pair[[
            "bandit_name", "method_group", "set_name", "original_trial_number",
            "confirmation_objective_score", "valid",
            "confirmation_median_of_seed_median_return_pct",
            "confirmation_mean_of_seed_median_return_pct",
            "confirmation_mean_of_seed_mean_return_pct",
            "confirmation_std_of_seed_median_return_pct",
            "confirmation_median_seed_val_first_median_return_pct",
            "confirmation_median_seed_val_second_median_return_pct",
            "confirmation_median_seed_min_half_median_return_pct",
            "confirmation_median_seed_half_return_gap_pct",
            "confirmation_median_max_drawdown_abs",
            "confirmation_median_profit_factor",
            "confirmation_median_total_trades",
            "memory_horizon_bars",
        ]])

        display(final_selection)
else:
    print("RUN_TS_CONFIRMATION=False; confirmation skipped.")


## 11. Optional: merge outputs from two computers

После завершения двух запусков скопируй папку с другого компьютера внутрь `stage2_optuna_state1_median_return_constraints_func_memory250_400_local`, затем выполни эту ячейку.

Эта ячейка объединяет основные `trial_results_all.csv`. Confirmation-файлы можно объединять отдельно через `ts_top3_confirmation_results.csv`, если оба запуска уже завершены.


In [ ]:
MERGE_ALL_LOCAL_RUNS = False

if MERGE_ALL_LOCAL_RUNS:
    merged_out_dir = OUTPUT_ROOT / "_merged_all_runs"
    merged_out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Merge all main Optuna trials.
    all_trial_files = sorted(OUTPUT_ROOT.glob("*/trial_results_all.csv"))
    print("Found trial files:", len(all_trial_files))
    if all_trial_files:
        merged = pd.concat([pd.read_csv(p) for p in all_trial_files], ignore_index=True)
        merged.to_csv(merged_out_dir / "merged_trial_results_all.csv", index=False)

        merged_ranking = (
            merged
            .sort_values(
                ["bandit_name", "method_group", *ranking_cols],
                ascending=[True, True, *ranking_ascending],
            )
            .reset_index(drop=True)
        )
        merged_ranking["rank_within_algorithm_method"] = (
            merged_ranking.groupby(["bandit_name", "method_group"]).cumcount() + 1
        )
        merged_best_per_pair = merged_ranking[merged_ranking["rank_within_algorithm_method"].eq(1)].copy()
        merged_best_per_pair.to_csv(merged_out_dir / "merged_best_trial_per_algorithm_method.csv", index=False)

        display(merged_best_per_pair[[
            "bandit_name", "method_group", "set_name", "trial_number", "valid",
            "median_of_seed_median_return_pct", "mean_of_seed_median_return_pct", "mean_of_seed_mean_return_pct",
            "median_max_drawdown_abs", "median_profit_factor", "median_total_trades",
            "memory_horizon_bars", "objective_score",
        ]])

    # 2) Merge TS confirmation outputs, if present.
    confirmation_files = sorted(OUTPUT_ROOT.glob("*/ts_top3_confirmation_results.csv"))
    print("Found TS confirmation files:", len(confirmation_files))
    if confirmation_files:
        merged_confirmation = pd.concat([pd.read_csv(p) for p in confirmation_files], ignore_index=True)
        merged_confirmation.to_csv(merged_out_dir / "merged_ts_top3_confirmation_results.csv", index=False)

    # 3) Merge final selected rows: TS rows use confirmation; UCB rows use Optuna.
    final_selection_files = sorted(OUTPUT_ROOT.glob("*/best_trial_per_algorithm_method_with_ts_confirmation.csv"))
    print("Found final selection files:", len(final_selection_files))
    if final_selection_files:
        merged_final_selection = pd.concat([pd.read_csv(p) for p in final_selection_files], ignore_index=True)
        merged_final_selection.to_csv(merged_out_dir / "merged_best_trial_per_algorithm_method_with_ts_confirmation.csv", index=False)
        display(merged_final_selection)
